# Занятие 9. Методы полей сил

In [ ]:
# 📥 Установка пакетов и библиотек
! pip install matplotlib
! pip install numpy
! pip install rdkit
! pip install parmed
! pip install scipy
! pip install seaborn
! pip install pandas


In [ ]:
# 🛠️ Загрузка необходимых пакетов в среду выполнения
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

from scipy.ndimage import gaussian_filter

from rdkit import Chem
from rdkit.Chem import Draw, AllChem

from matplotlib import animation
from IPython.display import HTML

# show() открывает 3D-просмотр структур и траекторий GROMACS прямо под ячейкой
REPO = "/home/jupyter/project/computational-chemistry-course"
sys.path.append(os.path.join(REPO, "tools"))
from md_tools import show

In [ ]:
# 𓅨 Инициализация GROMACS, PLUMED, AmberTools и PyRosetta
GMX = "/opt/gromacs/bin/gmx"                # GROMACS 2024.5 с патчем PLUMED в образе курса
PLUMED = "/opt/plumed/bin/plumed"
print("GROMACS найден:")
!/opt/gromacs/bin/gmx --version | head -1
!/opt/plumed/bin/plumed info --version

AMBERHOME = "/opt/ambertools"                # AmberTools в образе курса: antechamber, parmchk2, tleap
os.environ["AMBERHOME"] = AMBERHOME
os.environ["PATH"] = os.environ["PATH"] + ":" + os.path.join(AMBERHOME, "bin")   # в конце PATH: python окружения AmberTools не перекрывает системный
!which antechamber

import pyrosetta                             # PyRosetta предустановлена в образе курса
pyrosetta.init("-mute all")
print("PyRosetta найден:", pyrosetta.rosetta.utility.Version.version())


# 🏔️ Глава 1. Методы улучшенного сэмплинга

## 1.1 Проблема временных масштабов. Редкие события, барьеры

В занятии 8 мы получили карту Рамачандрана дипептида аланина за 1 нс динамики. Пептид за это время не выходил за пределы одной-двух областей, и карта выглядела как пара пятен. Это не ошибка моделирования, а его главное ограничение: молекулярная динамика честно воспроизводит движение системы, но делает это в реальном времени: при шаге интегрирования 2 фс 1 нс симуляции - это полмиллиона шагов.

Конформационные переходы - редкие события. Между состояниями лежат барьеры свободной энергии ΔG, и система большую часть времени колеблется на дне ямы, лишь изредка получая от тепловых флуктуаций энергию, достаточную для движения через барьер.

Отсюда два следствия. Во-первых, траектория, которая не успела пересечь барьеры, описывает не равновесие, а окрестность стартовой точки: распределение конформаций в ней зависит от того, откуда мы начали симуляцию. Во-вторых, из такой траектории нельзя получить свободную энергию, потому что заселённости состояний в ней не термодинамические. Чтобы увидеть это своими глазами, запустим три одинаковые динамики дипептида из трёх разных стартовых конформаций: β, αR и αL.

In [ ]:
main_path = os.path.join(REPO, "Занятия", "Занятие_9")
work_path = os.path.join(main_path, "ala2_starts")
os.makedirs(work_path, exist_ok=True)
os.chdir(work_path)

In [ ]:
# Стартовые точки на карте Рамачандрана: (φ₂, ψ₁) в градусах
STARTS = {
    "beta":   (-135.0, 135.0),   # β-лист
    "alphaR": ( -65.0, -40.0),   # правая α-спираль
    "alphaL": (  60.0,  45.0),   # левая α-спираль
}
SEQ = "NALA CALA"

# tleap строит цепь из библиотеки силового поля, impose изменяет двугранные углы.
#  ψ₁ = N₁-Cα₁-C₁-N₂
#  φ₂ = C₁-N₂-Cα₂-C₂  
for name, (phi, psi) in STARTS.items():
    os.makedirs(name, exist_ok=True)
    tleap_in = f"""\
source leaprc.protein.ff14SB
pep = sequence {{ {SEQ} }}
impose pep {{ 1 }} {{ {{ "N" "CA" "C" "N" {psi} }} }}
impose pep {{ 2 }} {{ {{ "C" "CA" "N" "C" {phi} }} }}
savepdb pep {name}/ala2.pdb
quit
"""
    with open(f"{name}/tleap.in", "w") as f:
        f.write(tleap_in)
    os.system(f"tleap -f {name}/tleap.in > {name}/tleap.log 2>&1")

In [ ]:
show("alphaR/ala2.pdb", "beta/ala2.pdb", "alphaL/ala2.pdb")

In [ ]:
# Минимизация энергии, NVT и NPT уравновешивание, моделирование в NPT ансамбле 
em_mdp = """\
define        = -DFLEXIBLE   ; гибкая вода только на минимизации
integrator    = steep        ; наискорейший спуск
emtol         = 100.0        ; критерий сходимости: максимальная сила на атоме, кДж/моль/нм
emstep        = 0.01         ; начальный шаг, нм
nsteps        = 5000         ; предельное число шагов

cutoff-scheme = Verlet       ; список соседей Верле с буфером
nstlist       = 10           ; список соседей обновляется каждые 10 шагов
coulombtype   = PME          ; электростатика по Эвальду
rcoulomb      = 1.0          ; радиус отсечки прямой части PME, нм
fourierspacing = 0.12        ; шаг сетки PME, нм
pme-order     = 4            ; кубическая интерполяция зарядов на сетку
vdwtype       = Cut-off      ; ЛД отсекается на rvdw
rvdw          = 1.0          ; радиус отсечки ЛД, нм
DispCorr      = EnerPres     ; хвостовая поправка к энергии и давлению
pbc           = xyz          ; периодические граничные условия по трём осям
"""
with open("em.mdp", "w") as f:
    f.write(em_mdp)

nvt_mdp = """\
integrator    = md           ; leap-frog
dt            = 0.002        ; 2 фс: связи с водородом зафиксированы
nsteps        = 10000        ; 10000 × 2 фс = 20 пс

nstxout-compressed = 0       ; траекторию не сохраняем
nstenergy     = 100          ; энергии в .edr каждые 0.2 пс
nstlog        = 1000         ; строка в .log каждые 2 пс

continuation  = no           ; первый шаг динамики: стартовое состояние из em.gro
constraints   = h-bonds      ; длины связей X-H пептида фиксированы; воду держит SETTLE из tip3p.itp
constraint-algorithm = lincs ; алгоритм LINCS для связей пептида

cutoff-scheme = Verlet       ; список соседей Верле с буфером
nstlist       = 10           ; список соседей обновляется каждые 10 шагов
coulombtype   = PME          ; электростатика по Эвальду
rcoulomb      = 1.0          ; радиус отсечки прямой части PME, нм
fourierspacing = 0.12        ; шаг сетки PME, нм
pme-order     = 4            ; кубическая интерполяция зарядов на сетку
vdwtype       = Cut-off      ; ЛД отсекается на rvdw
rvdw          = 1.0          ; радиус отсечки ЛД, нм
DispCorr      = EnerPres     ; хвостовая поправка к энергии и давлению
pbc           = xyz          ; периодические граничные условия по трём осям

tcoupl        = V-rescale    ; термостат Бусси
tc-grps       = System       ; одна группа: пептид и вода вместе
tau-t         = 0.1          ; время связи с термостатом, пс
ref-t         = 298.15       ; целевая температура, K
pcoupl        = no           ; объём фиксирован, ансамбль NVT

gen-vel       = yes          ; стартовые скорости из распределения Максвелла
gen-temp      = 298.15       ; при температуре 298.15 K
gen-seed      = -1           ; seed для случайных чисел
"""
with open("nvt.mdp", "w") as f:
    f.write(nvt_mdp)

npt_mdp = """\
integrator    = md           ; leap-frog
dt            = 0.002        ; 2 фс
nsteps        = 10000        ; 10000 × 2 фс = 20 пс

nstxout-compressed = 0       ; траекторию не сохраняем
nstenergy     = 100          ; энергии, объём и плотность в .edr каждые 0.2 пс
nstlog        = 1000         ; строка в .log каждые 2 пс

continuation  = yes          ; продолжение NVT: скорости и состояние термостата из nvt.cpt
constraints   = h-bonds      ; как в NVT
constraint-algorithm = lincs ; как в NVT

cutoff-scheme = Verlet       ; список соседей Верле с буфером
nstlist       = 10           ; список соседей обновляется каждые 10 шагов
coulombtype   = PME          ; электростатика по Эвальду
rcoulomb      = 1.0          ; радиус отсечки прямой части PME, нм
fourierspacing = 0.12        ; шаг сетки PME, нм
pme-order     = 4            ; кубическая интерполяция зарядов на сетку
vdwtype       = Cut-off      ; ЛД отсекается на rvdw
rvdw          = 1.0          ; радиус отсечки ЛД, нм
DispCorr      = EnerPres     ; хвостовая поправка к энергии и давлению
pbc           = xyz          ; периодические граничные условия по трём осям

tcoupl        = V-rescale    ; термостат Бусси, тот же, что в NVT
tc-grps       = System       ; термостатируемая группа - все атомы
tau-t         = 0.1          ; время связи с термостатом, пс
ref-t         = 298.15       ; целевая температура, K
pcoupl          = C-rescale  ; баростат, стохастическое масштабирование ячейки
pcoupltype      = isotropic  ; бокс масштабируется одинаково по трём осям
tau-p           = 2.0        ; время связи с баростатом, пс
ref-p           = 1.0        ; целевое давление, бар
compressibility = 4.5e-5     ; изотермическая сжимаемость воды при 298 K, 1/бар

gen-vel       = no           ; скорости продолжаются из nvt.cpt
"""
with open("npt.mdp", "w") as f:
    f.write(npt_mdp)

md_mdp = """\
integrator    = md           ; leap-frog
dt            = 0.002        ; 2 фс
nsteps        = 500000       ; 500000 × 2 фс = 1 нс

nstxout-compressed = 500     ; кадр в md.xtc каждую 1 пс: 1000 кадров для карты Рамачандрана
nstenergy     = 500          ; энергии, T, P, объём и плотность в .edr каждую 1 пс
nstlog        = 5000         ; строка в .log каждые 10 пс

continuation  = yes          ; продолжение NPT: скорости, термостат и баростат из npt.cpt
constraints   = h-bonds      ; как в NVT
constraint-algorithm = lincs ; как в NVT

cutoff-scheme = Verlet       ; список соседей Верле с буфером
nstlist       = 10           ; список соседей обновляется каждые 10 шагов
coulombtype   = PME          ; электростатика по Эвальду
rcoulomb      = 1.0          ; радиус отсечки прямой части PME, нм
fourierspacing = 0.12        ; шаг сетки PME, нм
pme-order     = 4            ; кубическая интерполяция зарядов на сетку
vdwtype       = Cut-off      ; ЛД отсекается на rvdw
rvdw          = 1.0          ; радиус отсечки ЛД, нм
DispCorr      = EnerPres     ; хвостовая поправка к энергии и давлению
pbc           = xyz          ; периодические граничные условия по трём осям

tcoupl        = V-rescale    ; термостат Бусси
tc-grps       = System       ; термостатируемая группа - все атомы
tau-t         = 0.1          ; время связи с термостатом, пс
ref-t         = 298.15       ; целевая температура, K
pcoupl          = C-rescale  ; баростат, тот же, что при уравновешивании
pcoupltype      = isotropic  ; бокс масштабируется одинаково по трём осям
tau-p           = 2.0        ; время связи с баростатом, пс
ref-p           = 1.0        ; целевое давление, бар
compressibility = 4.5e-5     ; изотермическая сжимаемость воды при 298 K, 1/бар

gen-vel       = no           ; скорости продолжаются из npt.cpt
"""
with open("md.mdp", "w") as f:
    f.write(md_mdp)

In [ ]:
# Четыре стадии динамики для каждой стартовой конформации
for name in STARTS:
    os.chdir(name)
    print(f"--- {name}")

    # Топология AMBER99SB-ILDN, кубический бокс 1.2 нм, вода TIP3P
    !/opt/gromacs/bin/gmx pdb2gmx -f ala2.pdb -o ala2.gro -p topol.top -ff amber99sb-ildn -water tip3p -ignh > pdb2gmx.log 2>&1
    !/opt/gromacs/bin/gmx editconf -f ala2.gro -o box.gro -c -d 1.2 -bt cubic > editconf.log 2>&1
    !/opt/gromacs/bin/gmx solvate -cp box.gro -cs spc216.gro -p topol.top -o solv.gro > solvate.log 2>&1

    # Минимизация энергии
    !/opt/gromacs/bin/gmx grompp -f ../em.mdp -c solv.gro -p topol.top -o em.tpr > grompp_em.out 2>&1
    !/opt/gromacs/bin/gmx mdrun -deffnm em -ntomp 4 > mdrun_em.out 2>&1

    # NVT уравновешивание
    !/opt/gromacs/bin/gmx grompp -f ../nvt.mdp -c em.gro -p topol.top -o nvt.tpr > grompp_nvt.out 2>&1
    !/opt/gromacs/bin/gmx mdrun -deffnm nvt -ntomp 4 > mdrun_nvt.out 2>&1

    # NPT уравновешивание
    !/opt/gromacs/bin/gmx grompp -f ../npt.mdp -c nvt.gro -t nvt.cpt -p topol.top -o npt.tpr > grompp_npt.out 2>&1
    !/opt/gromacs/bin/gmx mdrun -deffnm npt -ntomp 4 > mdrun_npt.out 2>&1

    # NPT моделирование 1 нс
    !/opt/gromacs/bin/gmx grompp -f ../md.mdp -c npt.gro -t npt.cpt -p topol.top -o md.tpr > grompp_md.out 2>&1
    !/opt/gromacs/bin/gmx mdrun -deffnm md -ntomp 4 > mdrun_md.out 2>&1

    # Траектория одного пептида без воды, целая молекула в центре бокса
    !printf "Protein\nProtein\n" | /opt/gromacs/bin/gmx trjconv -s md.tpr -f md.xtc -pbc mol -center -o md_mol.xtc > trjconv.log 2>&1

    os.chdir("..")


In [ ]:
# Карты Рамачандрана (φ₂, ψ₁) трёх траекторий из разных стартовых конформаций
SYSTEMS = {"beta": "старт β", "alphaR": "старт αR", "alphaL": "старт αL"}
colors = {"beta": "#1F77B4", "alphaR": "#D9622B", "alphaL": "#2E8B57"}

fig, axes = plt.subplots(1, 3, figsize=(12.5, 4.2), sharex=True, sharey=True)
for ax, (name, title) in zip(axes, SYSTEMS.items()):
    # Номера атомов ψ₁ (N₁-Cα₁-C₁-N₂) и φ₂ (C₁-N₂-Cα₂-C₂) по именам из ala2.gro
    gro = open(f"{name}/ala2.gro").read().splitlines()
    n = int(gro[1])
    idx = {(int(l[0:5]), l[10:15].strip()): i + 1 for i, l in enumerate(gro[2:2 + n])}   # (остаток, атом) → номер
    psi1 = [idx[1, "N"], idx[1, "CA"], idx[1, "C"], idx[2, "N"]]
    phi2 = [idx[1, "C"], idx[2, "N"], idx[2, "CA"], idx[2, "C"]]
    with open(f"{name}/dihedrals.ndx", "w") as f:
        f.write("[ psi1_phi2 ]\n" + " ".join(map(str, psi1 + phi2)) + "\n")
    os.system(f"echo 0 | /opt/gromacs/bin/gmx angle -f {name}/md_mol.xtc -n {name}/dihedrals.ndx -type dihedral -all -ov {name}/dihedrals.xvg > {name}/angle.log 2>&1")
    _, _, psi, phi = np.loadtxt(f"{name}/dihedrals.xvg", comments=("#", "@")).T   # время в пс, среднее, ψ₁, φ₂

    sns.kdeplot(x=phi, y=psi, ax=ax, fill=True, cmap=sns.light_palette(colors[name], as_cmap=True),
                levels=12, thresh=0.02, bw_adjust=0.7)
    ax.set_xlim(-180, 180)
    ax.set_ylim(-180, 180)
    ax.set_xticks([-180, -90, 0, 90, 180])
    ax.set_yticks([-180, -90, 0, 90, 180])
    ax.set_aspect("equal")
    ax.set_xlabel("φ, °")
    ax.set_title(title)
axes[0].set_ylabel("ψ, °")
plt.show()

## 1.2 Имитация отжига (Simulated annealing)

Время ожидания перехода через барьер растёт как $\tau \sim \exp\left(\frac{\Delta G}{k_\mathrm{B}T}\right)$, и в показателе экспоненты есть всего две величины, на которые мы можем повлиять. Отсюда два способа ускорить конформационные переходы:

1. **Повысить температуру.** При большей $T$ тепловые флуктуации чаще дают системе энергию, достаточную для движения через потенциальный барьер: при нагреве с 298 до 900 K показатель экспоненты уменьшается втрое, и барьер в 30 кДж/моль ощущается как барьер в 10 кДж/моль. 

2. **Изменить гамильтониан.** Можно оставить температуру прежней, но масштабировать взаимодействия так, чтобы барьеры стали ниже, например, умножить торсионный потенциал на λ < 1, ослабить электростатику, выключить часть взаимодействий. Экспонента та же, но изменяется $\Delta G$, а не $k_\mathrm{B}T$.

In [ ]:
# Частица на торсионном потенциале: обычная температура, нагрев и заниженный барьер
R = 8.314462618e-3                  # кДж/(моль·K)
A, B = 8.0, 4.0                     # V(φ) = A(1 + cos 3φ) + B(1 - cos φ), кДж/моль

def V(phi):
    return A * (1 + np.cos(3 * phi)) + B * (1 - np.cos(phi))

def force(phi):
    return 3 * A * np.sin(3 * phi) - B * np.sin(phi)     # -dV/dφ

RUNS = [("обычная МД, 298 K",        298.0, 1.0,     "#3B6EA8"),
        ("нагрев, 900 K",             900.0, 1.0,     "#DC602E"),
        ("298 K, барьер × 1/3",       298.0, 1 / 3,   "#6B8E23")]   # (заголовок, T, λ, цвет)

N_BALLS = 30                        # шариков в каждой панели
I = 0.5                             # момент инерции, а.е.м.·нм² (кДж/моль·пс²)
GAMMA = 5.0                         # трение термостата Ланжевена, 1/пс
DT = 0.002                          # шаг, пс
STEPS_PER_FRAME = 100               # 0.2 пс между кадрами анимации
N_FRAMES = 150                      # 30 пс всего

rng = np.random.default_rng(1)
c1 = np.exp(-GAMMA * DT)
c2 = np.sqrt(1 - c1 ** 2)

# Ланжевеновская динамика (схема BAOAB) для всех трёх систем сразу; старт - все шарики в левой яме
phi = np.full((len(RUNS), N_BALLS), np.deg2rad(-60.0))
kT = np.array([R * T for _, T, _, _ in RUNS])[:, None]
lam = np.array([l for _, _, l, _ in RUNS])[:, None]
v = rng.normal(0, np.sqrt(kT / I), phi.shape)
traj = np.empty((N_FRAMES, len(RUNS), N_BALLS))
for f in range(N_FRAMES):
    for _ in range(STEPS_PER_FRAME):
        v += 0.5 * DT * lam * force(phi) / I
        phi += 0.5 * DT * v
        v = c1 * v + c2 * np.sqrt(kT / I) * rng.normal(size=phi.shape)
        phi += 0.5 * DT * v
        v += 0.5 * DT * lam * force(phi) / I
    phi = (phi + np.pi) % (2 * np.pi) - np.pi              # периодичность: φ в (-180°, 180°]
    traj[f] = phi

# Рисунок: сверху потенциал и шарики, снизу накопленное распределение φ
grid = np.linspace(-np.pi, np.pi, 361)
deg = np.rad2deg(grid)
edges = np.linspace(-180, 180, 37)
centers = 0.5 * (edges[1:] + edges[:-1])
p_ref = np.exp(-V(grid) / (R * 298.0))
p_ref /= p_ref.sum() * (deg[1] - deg[0])                    # Больцман при 298 K для настоящего V, нормировка на 1

fig, axes = plt.subplots(2, 3, figsize=(12.5, 5.2), sharex=True, gridspec_kw={"height_ratios": [1.3, 1]})
balls, hists = [], []
for k, (title, T, l, color) in enumerate(RUNS):
    ax, axh = axes[0, k], axes[1, k]
    if l != 1.0:
        ax.plot(deg, V(grid), color="grey", lw=0.8, ls="--")   # настоящий потенциал для сравнения
    ax.plot(deg, l * V(grid), color="black", lw=1.2)
    balls.append(ax.scatter([], [], s=40, color=color, edgecolor="black", lw=0.4, zorder=3))
    ax.set_title(title)
    ax.set_ylim(-1, 25)
    axh.plot(deg, p_ref, color="black", lw=0.8, ls="--")
    hists.append(axh.stairs(np.zeros(len(centers)), edges, fill=True, color=color, alpha=0.6))
    axh.set_ylim(0, 1.6 * p_ref.max())
    axh.set_xlim(-180, 180)
    axh.set_xticks([-180, -90, 0, 90, 180])
    axh.set_xlabel("φ, °")
axes[0, 0].set_ylabel("V, кДж/моль")
axes[1, 0].set_ylabel("P(φ)")
time_text = axes[0, 0].text(0.03, 0.92, "", transform=axes[0, 0].transAxes)

def update(f):
    for k, (_, _, l, _) in enumerate(RUNS):
        x = traj[f, k]
        balls[k].set_offsets(np.column_stack([np.rad2deg(x), l * V(x) + 0.8]))   # шарик чуть над кривой
        H, _ = np.histogram(np.rad2deg(traj[:f + 1, k].ravel()), bins=edges, density=True)
        hists[k].set_data(H)
    time_text.set_text(f"t = {(f + 1) * STEPS_PER_FRAME * DT:.1f} пс")
    return balls + hists + [time_text]

anim = animation.FuncAnimation(fig, update, frames=N_FRAMES, interval=60, blit=False)
plt.close(fig)
with plt.rc_context({"savefig.dpi": 80, "animation.embed_limit": 40}):   # все 151 кадр, ~20 МБ в выводе
    html = anim.to_jshtml()
HTML(html)

### 1.2.1 Нагрев системы термостатом 

In [ ]:
main_path = os.path.join(REPO, "Занятия", "Занятие_9")
work_path = os.path.join(main_path, "ala2_starts", "anneal")
os.makedirs(work_path, exist_ok=True)
os.chdir(work_path)

In [ ]:
# Стартовое состояние - уравновешенная система β из раздела 1.1
!cp ../beta/npt.gro ../beta/npt.cpt ../beta/topol.top ../beta/ala2.gro .

In [ ]:
anneal_mdp = """\
integrator    = md           ; leap-frog
dt            = 0.002        ; 2 фс: связи с водородом зафиксированы
nsteps        = 500000       ; 500000 × 2 фс = 1 нс, четыре цикла нагрева

nstxout-compressed = 500     ; кадр в anneal.xtc каждую 1 пс
nstenergy     = 500          ; энергии и температура в .edr каждую 1 пс
nstlog        = 5000         ; строка в .log каждые 10 пс

continuation  = yes          ; продолжение NPT из раздела 1.1: скорости и термостат из npt.cpt
constraints   = h-bonds      ; длины связей X-H пептида фиксированы
constraint-algorithm = lincs ; алгоритм LINCS для связей пептида

cutoff-scheme = Verlet       ; список соседей Верле с буфером
nstlist       = 10           ; список соседей обновляется каждые 10 шагов
coulombtype   = PME          ; электростатика по Эвальду
rcoulomb      = 1.0          ; радиус отсечки прямой части PME, нм
fourierspacing = 0.12        ; шаг сетки PME, нм
pme-order     = 4            ; кубическая интерполяция зарядов на сетку
vdwtype       = Cut-off      ; ЛД отсекается на rvdw
rvdw          = 1.0          ; радиус отсечки ЛД, нм
DispCorr      = EnerPres     ; хвостовая поправка к энергии и давлению
pbc           = xyz          ; периодические граничные условия по трём осям

tcoupl        = V-rescale    ; термостат Бусси
tc-grps       = System       ; греем всю систему: пептид и воду
tau-t         = 0.1          ; время связи с термостатом, пс
ref-t         = 298.15       ; исходная температура, K; дальше её задаёт расписание отжига
pcoupl        = no           ; объём фиксирован, ансамбль NVT

annealing         = periodic                          ; расписание повторяется с периодом последней точки
annealing-npoints = 5                                 ; точек в расписании
annealing-time    = 0       50      100  200  250     ; время, пс
annealing-temp    = 298.15  298.15  500  500  298.15  ; целевая температура в эти моменты, K

gen-vel       = no           ; скорости продолжаются из npt.cpt
"""
with open("anneal.mdp", "w") as f:
    f.write(anneal_mdp)

!/opt/gromacs/bin/gmx grompp -f anneal.mdp -c npt.gro -t npt.cpt -p topol.top -o anneal.tpr > grompp_anneal.out 2>&1
!/opt/gromacs/bin/gmx mdrun -deffnm anneal -ntomp 4 > mdrun_anneal.out 2>&1

# Траектория одного пептида и температура системы по времени
!printf "Protein\nProtein\n" | /opt/gromacs/bin/gmx trjconv -s anneal.tpr -f anneal.xtc -pbc mol -center -o anneal_mol.xtc > trjconv.log 2>&1
!echo Temperature | /opt/gromacs/bin/gmx energy -f anneal.edr -o temperature.xvg > energy.log 2>&1

In [ ]:
# Углы ψ и φ по траектории отжига и температура в момент каждого кадра
gro = open("ala2.gro").read().splitlines()
n = int(gro[1])
idx = {(int(l[0:5]), l[10:15].strip()): i + 1 for i, l in enumerate(gro[2:2 + n])}   # (остаток, атом) → номер
psi1 = [idx[1, "N"], idx[1, "CA"], idx[1, "C"], idx[2, "N"]]
phi2 = [idx[1, "C"], idx[2, "N"], idx[2, "CA"], idx[2, "C"]]
with open("dihedrals.ndx", "w") as f:
    f.write("[ psi1_phi2 ]\n" + " ".join(map(str, psi1 + phi2)) + "\n")
!echo 0 | /opt/gromacs/bin/gmx angle -f anneal_mol.xtc -n dihedrals.ndx -type dihedral -all -ov dihedrals.xvg > angle.log 2>&1

t, _, psi, phi = np.loadtxt("dihedrals.xvg", comments=("#", "@")).T     # время в пс, среднее, ψ₁, φ₂
t_T, T = np.loadtxt("temperature.xvg", comments=("#", "@")).T
T_frames = np.interp(t, t_T, T)                                       # температура системы в момент кадра

fig = plt.figure(figsize=(12.5, 4.2))
gs = fig.add_gridspec(2, 2, width_ratios=[2.2, 1], height_ratios=[1, 1.4], wspace=0.25, hspace=0.08)
ax_T = fig.add_subplot(gs[0, 0])
ax_phi = fig.add_subplot(gs[1, 0], sharex=ax_T)
ax_r = fig.add_subplot(gs[:, 1])

ax_T.plot(t_T / 1000, T, color="black", lw=0.6)
ax_T.set_ylabel("T, K")
ax_T.tick_params(labelbottom=False)

ax_phi.scatter(t / 1000, phi, c=T_frames, cmap="coolwarm", vmin=298, vmax=500, s=3)
ax_phi.axhline(0, color="grey", lw=0.6, ls="--")                      # φ₂ = 0: граница αR/β и αL
ax_phi.set_ylim(-180, 180)
ax_phi.set_yticks([-180, -90, 0, 90, 180])
ax_phi.set_ylabel("φ$_2$, °")
ax_phi.set_xlabel("t, нс")
ax_phi.set_xlim(0, t[-1] / 1000)

sc = ax_r.scatter(phi, psi, c=T_frames, cmap="coolwarm", vmin=298, vmax=500, s=3)
ax_r.set_xlim(-180, 180)
ax_r.set_ylim(-180, 180)
ax_r.set_xticks([-180, -90, 0, 90, 180])
ax_r.set_yticks([-180, -90, 0, 90, 180])
ax_r.set_aspect("equal")
ax_r.set_xlabel("φ, °")
ax_r.set_ylabel("ψ, °")
fig.colorbar(sc, ax=ax_r, shrink=0.8, label="T, K")
plt.show()

Однако оба подхода имеют существенный недостаток: траектория и распределение конформаций, полученные при повышенной температуре или при модифицированном гамильтониане, не соответствуют исходной системе, например, при $T=298K$. Поэтому простого ускорения динамики недостаточно, необходимо каким то образом использовать полученную траекторию, сохраняя возможность восстановить распределение исходной системы. 

### 1.2.2 Молекулярная динамика с обменом репликами

При моделировании нескольких реплик при различной температуре в методе REMD (Replica Exchange Molecular Dynamics) попытка обмена между соседними репликами осуществляется через равные промежутки времени, и обмен принимается с вероятностью, определяемой по критерию Метрополиса (о нём подробнее в разделе 2.2): 

$$P_{acc} = \min\left(1,\ \exp\left[(\beta_i - \beta_j)(U_i - U_j)\right]\right), \qquad \beta = \frac{1}{k_BT}$$

где $U_i$ и $U_j$ - потенциальные энергии конфигураций в репликах $i$ и $j$. После обмена скорости масштабируются на $\sqrt{\dfrac{T_{new}}{T_{old}}}$, чтобы кинетическая энергия соответствовала новой температуре. Такой критерий сохраняет детальный баланс в расширенном ансамбле всех реплик, поэтому каждая реплика остаётся равновесной при своей температуре. Горячие реплики преодолевают барьеры, а холодная получает от них новые конформации.

In [ ]:
# Обмен репликами по температуре
T_rep = np.array([298.0, 420.0, 600.0, 900.0])              # температуры реплик, K
colors = ["#3B6EA8", "#6B8E23", "#E0A030", "#DC602E"]      # от холодной реплики к горячей
M = len(T_rep)
kT = (R * T_rep)[:, None]

x = np.full((M, N_BALLS), np.deg2rad(-60.0))
v = rng.normal(0, np.sqrt(kT / I), x.shape)
home = np.repeat(np.arange(M)[:, None], N_BALLS, axis=1)   # из какой реплики пришла конфигурация
n_exch = np.zeros(M - 1, dtype=int)                          # принятых обменов в каждой паре соседей
traj = np.empty((N_FRAMES + 1, M, N_BALLS))                  # кадр 0 - стартовое состояние при t = 0
homes = np.empty((N_FRAMES + 1, M, N_BALLS), dtype=int)
exch = np.empty((N_FRAMES + 1, M - 1), dtype=int)            # накопленное число обменов к каждому кадру
traj[0], homes[0], exch[0] = x, home, n_exch
for f in range(1, N_FRAMES + 1):
    for _ in range(STEPS_PER_FRAME):                         # BAOAB в каждой реплике
        v += 0.5 * DT * force(x) / I
        x += 0.5 * DT * v
        v = c1 * v + c2 * np.sqrt(kT / I) * rng.normal(size=x.shape)
        x += 0.5 * DT * v
        v += 0.5 * DT * force(x) / I
    x = (x + np.pi) % (2 * np.pi) - np.pi
    for i in range(f % 2, M - 1, 2):                         # обмен соседей: чётные и нечётные пары по очереди
        j = i + 1
        delta = (1 / (R * T_rep[i]) - 1 / (R * T_rep[j])) * (V(x[i]) - V(x[j]))
        acc = rng.random(N_BALLS) < np.exp(delta)            # Метрополис: min(1, exp[(β_i − β_j)(U_i − U_j)])
        n_exch[i] += acc.sum()
        x[i, acc], x[j, acc] = x[j, acc].copy(), x[i, acc].copy()
        v[i, acc], v[j, acc] = v[j, acc] * np.sqrt(T_rep[i] / T_rep[j]), v[i, acc] * np.sqrt(T_rep[j] / T_rep[i])
        home[i, acc], home[j, acc] = home[j, acc].copy(), home[i, acc].copy()
    traj[f], homes[f], exch[f] = x, home, n_exch

fig, axes = plt.subplots(2, M, figsize=(12.5, 5.2), sharex=True, gridspec_kw={"height_ratios": [1.3, 1]})
balls, hists, titles = [], [], []
for k in range(M):
    ax, axh = axes[0, k], axes[1, k]
    ax.plot(deg, V(grid), color="black", lw=1.2)
    balls.append(ax.scatter([], [], s=40, edgecolor="black", lw=0.4, zorder=3))
    titles.append(ax.set_title("", color=colors[k]))
    ax.set_ylim(-1, 25)
    ax.set_xlim(-180, 180)
    p = np.exp(-V(grid) / (R * T_rep[k]))
    axh.plot(deg, p / (p.sum() * (deg[1] - deg[0])), color="black", lw=0.8, ls="--")   # Больцман при T реплики
    hists.append(axh.stairs(np.zeros(len(edges) - 1), edges, fill=True, color=colors[k], alpha=0.6))
    axh.set_xlim(-180, 180)
    axh.set_ylim(0, 0.03)
    axh.set_xticks([-180, -90, 0, 90, 180])
    axh.set_xlabel("φ, °")
axes[0, 0].set_ylabel("V, кДж/моль")
axes[1, 0].set_ylabel("P(φ)")
time_text = axes[0, 0].text(0.03, 0.92, "", transform=axes[0, 0].transAxes)

def update(f):
    for k in range(M):
        balls[k].set_offsets(np.column_stack([np.rad2deg(traj[f, k]), V(traj[f, k]) + 0.8]))
        balls[k].set_facecolor([colors[h] for h in homes[f, k]])
        hists[k].set_data(np.histogram(np.rad2deg(traj[:f + 1, k].ravel()), bins=edges, density=True)[0])
        counts = ([str(exch[f, k - 1])] if k > 0 else []) + ([str(exch[f, k])] if k < M - 1 else [])   # с левой и правой соседкой
        titles[k].set_text(f"{T_rep[k]:.0f} K (обменов: {', '.join(counts)})")
    time_text.set_text(f"t = {f * STEPS_PER_FRAME * DT:.1f} пс")
    return balls + hists + titles + [time_text]

anim = animation.FuncAnimation(fig, update, frames=N_FRAMES + 1, interval=60, blit=False)
plt.close(fig)
with plt.rc_context({"savefig.dpi": 80, "animation.embed_limit": 40}):   # все 151 кадр, ~20 МБ в выводе
    html = anim.to_jshtml()
HTML(html)

### 1.2.3 Методы эффективного гамильтониана

Альетрнативный путь - не нагревать систему, а менять гамильтониан. Здесь работает аналогичная логика обмена репликами: несколько копий системы при одной температуре, но с разными множителями λ перед потенциалом, от λ = 1 (настоящая система) до $λ_min$, где барьеры в несколько раз ниже. Соседние реплики время от времени обмениваются конфигурациями. Критерий обмена выводится точно так же, как в REMD, только теперь различаются не температуры, а потенциалы:

$$P_{acc} = \min\left(1,\ \exp\left[\beta\,(\lambda_i - \lambda_j)\,(V(x_i) - V(x_j))\right]\right)$$

Скорости после обмена масштабировать не нужно: температура у всех реплик одна.

In [ ]:
# Обмен репликами по гамильтониану: все реплики при 298 K, различается множитель λ перед потенциалом
lam_rep = np.array([1.0, 0.7, 0.5, 0.33])                   # λ реплик: от настоящего потенциала к сглаженному
colors = ["#3B6EA8", "#6B8E23", "#E0A030", "#DC602E"]       # от λ = 1 к самой «мягкой» реплике
T = 298.0
kT = R * T
M = len(lam_rep)
lam = lam_rep[:, None]

x = np.full((M, N_BALLS), np.deg2rad(-60.0))
v = rng.normal(0, np.sqrt(kT / I), x.shape)                 # одна температура для всех реплик
home = np.repeat(np.arange(M)[:, None], N_BALLS, axis=1)    # из какой реплики пришла конфигурация
n_exch = np.zeros(M - 1, dtype=int)
traj = np.empty((N_FRAMES + 1, M, N_BALLS))
homes = np.empty((N_FRAMES + 1, M, N_BALLS), dtype=int)
exch = np.empty((N_FRAMES + 1, M - 1), dtype=int)
traj[0], homes[0], exch[0] = x, home, n_exch
for f in range(1, N_FRAMES + 1):
    for _ in range(STEPS_PER_FRAME):                          # BAOAB в каждой реплике с силой λ·F
        v += 0.5 * DT * lam * force(x) / I
        x += 0.5 * DT * v
        v = c1 * v + c2 * np.sqrt(kT / I) * rng.normal(size=x.shape)
        x += 0.5 * DT * v
        v += 0.5 * DT * lam * force(x) / I
    x = (x + np.pi) % (2 * np.pi) - np.pi
    for i in range(f % 2, M - 1, 2):                          # обмен соседей: чётные и нечётные пары по очереди
        j = i + 1
        delta = (lam_rep[i] - lam_rep[j]) * (V(x[i]) - V(x[j])) / kT
        acc = rng.random(N_BALLS) < np.exp(delta)             # Метрополис: min(1, exp[β(λ_i − λ_j)(V_i − V_j)])
        n_exch[i] += acc.sum()
        x[i, acc], x[j, acc] = x[j, acc].copy(), x[i, acc].copy()   # скорости не трогаем: температура одна
        home[i, acc], home[j, acc] = home[j, acc].copy(), home[i, acc].copy()
    traj[f], homes[f], exch[f] = x, home, n_exch

fig, axes = plt.subplots(2, M, figsize=(12.5, 5.2), sharex=True, gridspec_kw={"height_ratios": [1.3, 1]})
balls, hists, titles = [], [], []
for k in range(M):
    ax, axh = axes[0, k], axes[1, k]
    if lam_rep[k] != 1.0:
        ax.plot(deg, V(grid), color="grey", lw=0.8, ls="--")   # настоящий потенциал для сравнения
    ax.plot(deg, lam_rep[k] * V(grid), color="black", lw=1.2)
    balls.append(ax.scatter([], [], s=40, edgecolor="black", lw=0.4, zorder=3))
    titles.append(ax.set_title("", color=colors[k]))
    ax.set_ylim(-1, 25)
    ax.set_xlim(-180, 180)
    p = np.exp(-lam_rep[k] * V(grid) / kT)
    axh.plot(deg, p / (p.sum() * (deg[1] - deg[0])), color="black", lw=0.8, ls="--")   # Больцман для λV при 298 K
    hists.append(axh.stairs(np.zeros(len(edges) - 1), edges, fill=True, color=colors[k], alpha=0.6))
    axh.set_xlim(-180, 180)
    axh.set_ylim(0, 0.03)
    axh.set_xticks([-180, -90, 0, 90, 180])
    axh.set_xlabel("φ, °")
axes[0, 0].set_ylabel("λV, кДж/моль")
axes[1, 0].set_ylabel("P(φ)")
fig.subplots_adjust(left=0.06, right=0.99, top=0.93, bottom=0.1, wspace=0.28, hspace=0.12)   # заголовки не наезжают
time_text = axes[0, 0].text(0.03, 0.92, "", transform=axes[0, 0].transAxes)

def update(f):
    for k in range(M):
        balls[k].set_offsets(np.column_stack([np.rad2deg(traj[f, k]), lam_rep[k] * V(traj[f, k]) + 0.8]))
        balls[k].set_facecolor([colors[h] for h in homes[f, k]])
        hists[k].set_data(np.histogram(np.rad2deg(traj[:f + 1, k].ravel()), bins=edges, density=True)[0])
        counts = ([str(exch[f, k - 1])] if k > 0 else []) + ([str(exch[f, k])] if k < M - 1 else [])
        titles[k].set_text(f"λ = {lam_rep[k]:.2f} (обменов: {', '.join(counts)})")
    time_text.set_text(f"t = {f * STEPS_PER_FRAME * DT:.1f} пс")
    return balls + hists + titles + [time_text]

anim = animation.FuncAnimation(fig, update, frames=N_FRAMES + 1, interval=60, blit=False)
plt.close(fig)
with plt.rc_context({"savefig.dpi": 80, "animation.embed_limit": 40}):   # все 151 кадр, ~20 МБ в выводе
    html = anim.to_jshtml()
HTML(html)

Для одной частицы оба способа неразличимы: распределение Больцмана зависит только от отношения $V/k_BT$, и реплика с λ = 0.33 при 298 K ведёт себя ровно как реплика при 900 K с настоящим потенциалом. Разница появляется, когда в системе есть растворитель. Изменение температуры термостата нагревает всю систему, и число реплик, нужное для разумной вероятности обмена, растёт как корень из числа степеней свободы всей системы. Почти вся вычислительная работа уходит на барьеры воды, которые нам не мешают.

Масштабировать потенциал можно выборочно. В методе REST2 (Replica Exchange with Solute Tempering; [Wang, Friesner, Berne, 2011](https://doi.org/10.1021/jp204407d)) множитель λ применяется только к взаимодействиям внутри биополимера, а взаимодействие биополимер-вода масштабируется как √λ, из-за чего вода остаётся холодной во всех репликах. Тогда число реплик определяется только размером биополимера, и для дипептида аланина хватает четырёх-шести копий вместо десятков. Это тот же обмен репликами по гамильтониану, что в анимации выше, только λ стоит не перед всем потенциалом, а перед его частью.

Обмен репликами в любом варианте требует параллельного запуска копий (`mdrun -multidir -replex`) и сборки GROMACS с MPI, поэтому в ноутбуке мы его не запускаем.

## 1.3 Метадинамика

Имитация отжига и обмен репликами ускоряют переходы через барьеры, но делают это вслепую: они нагревают систему или ослабляют весь гамильтониан, не зная, где лежат барьеры и куда система должна перейти. Метадинамика ([Laio, Parrinello, 2002](https://doi.org/10.1073/pnas.202427399)) работает иначе. Мы выбираем одну или несколько коллективных переменных $s$, и во время динамики через каждые N шагов добавляем к текущей координате $s(t)$ небольшой смещающий потенциал:

$$V_b(s, t) = \sum_{t' < t} h \exp\left(-\frac{(s - s(t'))^2}{2\sigma^2}\right)$$

который постепенно заполняет яму, где находится система. Когда яма достаточно заполнена, система может перейти в соседнюю яму и заполнение продолжается уже там. Метадинамика работает как память о посещённых состояниях, поскольку системе становится энергетически невыгодно возвращаться к координатам коллективной переменной, где она уже была, и система вынуждена исследовать новые области. Когда все ямы заполнены, суммарный потенциал $V + V_b$ становится плоским, а свободная энергия может быть восстановлена как смещение с обратным знаком:

$$F(s) \approx -V_b(s, t \to \infty).$$

В обычной метадинамической симуляции все холмы одинаковой высоты, поэтому оценка $F$ колеблется вокруг правильного ответа и не сходится. В well-tempered метадинамике ([Barducci, Bussi, Parrinello, 2008](https://doi.org/10.1103/PhysRevLett.100.020603)) высота каждого нового холма уменьшается как $h\, e^{-V_b(s)/k_B\Delta T}$. Там, где смещение уже велико, холмы становятся ниже, и смещение плавно сходится к $F(s) = -\frac{\gamma}{\gamma - 1} V_b(s)$, где $\gamma = (T + \Delta T)/T$ - фактор смещения.


In [ ]:
# Обычная МД и Метадинамика
T = 298.0                           # температура всех трёх симуляций, K
kT = R * T

H0 = 1.2                            # высота холма, кДж/моль
SIGMA = np.deg2rad(20.0)            # ширина холма по φ
PACE = 100                          # холм каждые 100 шагов = 0.2 пс, один на кадр
GAMMA_WT = 10.0                     # фактор смещения well-tempered: (T + ΔT)/T

RUNS = [("обычная МД",            "md",  "#3B6EA8"),
        ("метадинамика",          "mtd", "#DC602E"),
        (f"well-tempered, γ = {GAMMA_WT:.0f}", "wt", "#6B8E23")]   # (заголовок, режим, цвет)

def wrap(a):
    return (a + np.pi) % (2 * np.pi) - np.pi

def hill(phi, s, h):
    """Один холм высоты h с центром s и его сила -dG/dφ, с учётом периодичности"""
    d = wrap(phi - s)
    g = h * np.exp(-d ** 2 / (2 * SIGMA ** 2))
    return g, g * d / SIGMA ** 2

M = len(RUNS)

# Ланжевеновская динамика (BAOAB); старт - левая яма. Холмы храним как центры и высоты
x = np.full(M, np.deg2rad(-60.0))
v = rng.normal(0, np.sqrt(kT / I), M)
hill_c = [[] for _ in RUNS]
hill_h = [[] for _ in RUNS]

def bias_force(k, phi):
    if not hill_c[k]:
        return 0.0
    return hill(phi, np.array(hill_c[k]), np.array(hill_h[k]))[1].sum()

def bias_value(k, phi):
    if not hill_c[k]:
        return 0.0
    return hill(phi, np.array(hill_c[k]), np.array(hill_h[k]))[0].sum()

def total_force(x):
    return np.array([force(x[k]) + bias_force(k, x[k]) for k in range(M)])

traj = np.empty((N_FRAMES + 1, M))
traj[0] = x
step = 0
for f in range(1, N_FRAMES + 1):
    for _ in range(STEPS_PER_FRAME):
        v += 0.5 * DT * total_force(x) / I
        x += 0.5 * DT * v
        v = c1 * v + c2 * np.sqrt(kT / I) * rng.normal(size=M)
        x += 0.5 * DT * v
        x = wrap(x)
        v += 0.5 * DT * total_force(x) / I
        step += 1
        if step % PACE == 0:                               # кладём холм в текущую точку
            for k, (_, mode, _) in enumerate(RUNS):
                if mode == "md":
                    continue
                h = H0
                if mode == "wt":                           # well-tempered: там, где смещение уже велико, холмы ниже
                    h = H0 * np.exp(-bias_value(k, x[k]) / (kT * (GAMMA_WT - 1)))
                hill_c[k].append(x[k])
                hill_h[k].append(h)
    traj[f] = x

# Накопленное смещение V_b на сетке после n холмов: cum[k][n]
cum = []
for k in range(M):
    c = np.zeros((len(hill_c[k]) + 1, len(grid)))
    for n, (s, h) in enumerate(zip(hill_c[k], hill_h[k])):
        c[n + 1] = c[n] + hill(grid, s, h)[0]
    cum.append(c)

def free_energy(k, f):
    """Оценка F(φ): из гистограммы для МД, из смещения для метадинамики; минимум в 0"""
    mode = RUNS[k][1]
    if mode == "md":
        P, _ = np.histogram(np.rad2deg(traj[:f + 1, k]), bins=72, range=(-180, 180))
        F = np.full(len(grid), np.nan)
        idx = np.clip(((deg + 180) / 5).astype(int), 0, 71)
        ok = P[idx] > 0
        F[ok] = -kT * np.log(P[idx][ok])
    else:
        n = min(f, len(hill_h[k]))
        F = -cum[k][n] * (1 if mode == "mtd" else GAMMA_WT / (GAMMA_WT - 1))
    return F - np.nanmin(F)

F_true = V(grid) - V(grid).min()

fig, axes = plt.subplots(2, M, figsize=(12.5, 5.2), sharex=True, gridspec_kw={"height_ratios": [1.3, 1]})
ests, balls, titles, layers = [], [], [], [[] for _ in range(M)]
for k, (title, mode, color) in enumerate(RUNS):
    ax, axf = axes[0, k], axes[1, k]
    ax.plot(deg, V(grid), color="black", lw=1.2)
    balls.append(ax.scatter([], [], s=50, color=color, edgecolor="black", lw=0.4, zorder=5))
    titles.append(ax.set_title(title))
    ax.set_ylim(-1, 50)
    axf.plot(deg, F_true, color="black", lw=0.8, ls="--")                 # точный ответ
    ests.append(axf.plot(deg, F_true * np.nan, color=color, lw=1.5)[0])
    axf.set_ylim(-2, 25)
    axf.set_xlim(-180, 180)
    axf.set_xticks([-180, -90, 0, 90, 180])
    axf.set_xlabel("φ, °")
axes[0, 0].set_ylabel("V + V$_b$, кДж/моль")
axes[1, 0].set_ylabel("F(φ), кДж/моль")
time_text = axes[0, 0].text(0.03, 0.92, "", transform=axes[0, 0].transAxes)

def update(f):
    for k, (title, mode, color) in enumerate(RUNS):
        ax = axes[0, k]
        for a in layers[k]:
            a.remove()
        layers[k] = []
        xb = traj[f, k]
        top = V(xb)
        if mode != "md":
            n = min(f, len(hill_h[k]))
            base = V(grid)
            layers[k].append(ax.fill_between(deg, base, base + cum[k][n], color=color, alpha=0.25, lw=0))   # вся «насыпь»
            for m in range(10, n, 10):                                   # слои по 10 холмов, как в осадочной породе
                layers[k].append(ax.plot(deg, base + cum[k][m], color=color, lw=0.4, alpha=0.6)[0])
            if n > 0:                                                    # последний холм - темнее
                layers[k].append(ax.fill_between(deg, base + cum[k][n - 1], base + cum[k][n], color=color, alpha=0.9, lw=0))
                layers[k].append(ax.plot(deg, base + cum[k][n], color="black", lw=1.0)[0])
            top += np.interp(np.rad2deg(xb), deg, cum[k][n])
            titles[k].set_text(f"{title} (холмов: {n})")
        balls[k].set_offsets([[np.rad2deg(xb), top + 1.2]])                 # шарик чуть над поверхностью
        ests[k].set_ydata(free_energy(k, f))
    time_text.set_text(f"t = {f * STEPS_PER_FRAME * DT:.1f} пс")
    return balls + ests + titles + [time_text]

anim = animation.FuncAnimation(fig, update, frames=N_FRAMES + 1, interval=80, blit=False)
plt.close(fig)
with plt.rc_context({"savefig.dpi": 80, "animation.embed_limit": 40}):   # все 151 кадр, ~20 МБ в выводе
    html = anim.to_jshtml()
HTML(html)

### 1.3.1 Коллективные переменные и поверхность свободной энергии

Конфигурация дипептида в воде - это координаты нескольких тысяч атомов, но для описания конформационных изменений дипептида достаточно двух двугранных углов φ и ψ. Такие функции координат s(R), сводящие 3N-мерное пространство к одной-двум переменным, называют **коллективными переменными** (collective variables, CV). Типичные CV - расстояние между атомами или центрами масс, валентный и двугранный углы, координационное число, радиус гирации, RMSD от опорной структуры.

Распределение вероятности CV в равновесном ансамбле P(s) задаёт **свободную энергию** вдоль этой переменной:

$$F(s) = -k_BT\ln P(s) + C$$

Константа C не важна: физический смысл имеют только разности. Отношение заселённостей двух состояний A и B переводится в разность свободных энергий:

$$\Delta F_{A\to B} = -k_BT\ln\frac{P_B}{P_A}$$

Формула верна только для **равновесного** P(s).

In [ ]:
main_path = os.path.join(REPO, "Занятия", "Занятие_9")
work_path = os.path.join(main_path, "ala2_starts")
os.chdir(work_path)

In [ ]:
# Поверхность свободной энергии F(φ, ψ) = -kT ln P по гистограммам траекторий из раздела 1.1
R = 8.314462618e-3                 # газовая постоянная, кДж/(моль·K)
T = 298.15                         # температура моделирования, K
kT = R * T                         # ≈ 2.48 кДж/моль
F_MAX = 10                         # верх цветовой шкалы, кДж/моль
BIN = 5                            # ячейка гистограммы, °
SMOOTH = 1.0                       # сглаживание гистограммы гауссианой, в ячейках

SYSTEMS = {"beta": "старт β", "alphaR": "старт αR", "alphaL": "старт αL"}
edges = np.arange(-180, 180 + BIN, BIN)
centers = 0.5 * (edges[1:] + edges[:-1])

def fes(phi, psi):
    H, _, _ = np.histogram2d(phi, psi, bins=[edges, edges])
    H = gaussian_filter(H, SMOOTH, mode="wrap")              # углы периодичны: края карты соседствуют
    P = np.where(H > 0.1, H / H.sum(), np.nan)               # ячейки без кадров - данных нет
    F = -kT * np.log(P)
    return F - np.nanmin(F)                                  # ноль - самый глубокий минимум

data = {}
for name in SYSTEMS:
    _, _, psi, phi = np.loadtxt(f"{name}/dihedrals.xvg", comments=("#", "@")).T   # время, среднее, ψ, φ
    data[name] = (phi, psi)
data["all"] = tuple(np.concatenate([data[n][k] for n in SYSTEMS]) for k in (0, 1))
titles = dict(SYSTEMS, all="три траектории вместе")
levels = np.arange(0, F_MAX + 1, 1)

fig, axes = plt.subplots(1, 4, figsize=(12.5, 3.0), sharex=True, sharey=True, constrained_layout=True)
for ax, name in zip(axes, titles):
    F = fes(*data[name])
    im = ax.contourf(centers, centers, F.T, levels=levels, cmap="RdYlBu_r", extend="max")   # .T: строки - ψ
    ax.contour(centers, centers, F.T, levels=levels[::2], colors="black", linewidths=0.3)
    ax.set_xlim(-180, 180)
    ax.set_ylim(-180, 180)
    ax.set_xticks([-180, -90, 0, 90, 180])
    ax.set_yticks([-180, -90, 0, 90, 180])
    ax.set_aspect("equal")
    ax.set_xlabel("φ, °")
    ax.set_title(titles[name])
axes[0].set_ylabel("ψ, °")
fig.colorbar(im, ax=axes[-1], fraction=0.05, pad=0.08, ticks=levels[::2], label="F, кДж/моль")
plt.show()

Хорошая коллективная переменная должна удовлетворять двум требованиям:

1. **Различать состояния.** Конформации, которые мы считаем разными (αR, β, αL), должны иметь разные значения коллективной переменной.
2. **Описывать медленное движение.** Барьер, который мешает сэмплингу, должен лежать вдоль коллективной переменной. 

### 1.3.2 Запуск метадинамических симуляций

Метадинамику в GROMACS считает не сам `mdrun`, а подключаемая библиотека [PLUMED](https://www.plumed.org) ([Tribello et al., 2014](https://doi.org/10.1016/j.cpc.2013.09.018); [The PLUMED consortium, 2019](https://doi.org/10.1038/s41592-019-0506-8)). Для этого GROMACS должен быть собран с поддержкой PLUMED (патч при сборке, в новых версиях - флаг `-DGMX_USE_PLUMED`), а при запуске добавляется ключ `mdrun -plumed plumed.dat`.

Файл `plumed.dat` описывает расчёт на собственном языке PLUMED: каждая строка - действие с меткой и аргументами. Нам нужны три действия:

- `TORSION ATOMS=...` - коллективные переменные, двугранные углы φ и ψ по номерам атомов. PLUMED считает их в радианах.
- `METAD ARG=phi,psi ...` - сама метадинамика: по каким CV добавлять холмы (`ARG`), как часто (`PACE`, в шагах), какой высоты (`HEIGHT`, кДж/моль) и ширины (`SIGMA`, в единицах CV), фактор смещения well-tempered (`BIASFACTOR` = γ) и температура (`TEMP`). Сумму холмов PLUMED хранит на сетке (`GRID_MIN`, `GRID_MAX`), иначе на каждом шаге пришлось бы суммировать тысячи гауссиан. Все положенные холмы записываются в файл `HILLS`.
- `PRINT ARG=... STRIDE=... FILE=COLVAR` - значения CV и смещения через заданное число шагов для анализа. 

Параметры подбираются по простым правилам. Ширина холма σ - порядка размаха колебаний CV внутри одной долины: для двугранных углов это 15–20°, то есть ~0.35 рад. Высота - доля k_BT, у нас 1.2 кДж/моль ≈ 0.5 k_BT, холмы кладём каждую пикосекунду. Фактор смещения γ подбирают по высоте барьера: эффективная температура T + ΔT = γT, при γ = 6 это около 1800 K, и барьер в 30 кДж/моль по φ становится проходимым за наносекунды. 

In [ ]:
main_path = os.path.join(REPO, "Занятия", "Занятие_9")
work_path = os.path.join(main_path, "ala2_starts", "metad")
os.makedirs(work_path, exist_ok=True)
os.chdir(work_path)

# Стартовое состояние - уравновешенная система β из раздела 1.1
!cp ../beta/npt.gro ../beta/npt.cpt ../beta/topol.top ../beta/ala2.gro .

In [ ]:
# Номера атомов ψ (N₁-Cα₁-C₁-N₂) и φ (C₁-N₂-Cα₂-C₂) по именам из ala2.gro
gro = open("ala2.gro").read().splitlines()
n = int(gro[1])
idx = {(int(l[0:5]), l[10:15].strip()): i + 1 for i, l in enumerate(gro[2:2 + n])}   # (остаток, атом) → номер
psi1 = [idx[1, "N"], idx[1, "CA"], idx[1, "C"], idx[2, "N"]]
phi2 = [idx[1, "C"], idx[2, "N"], idx[2, "CA"], idx[2, "C"]]

plumed_dat = f"""\
# Коллективные переменные: двугранные углы, в радианах
phi: TORSION ATOMS={",".join(map(str, phi2))}
psi: TORSION ATOMS={",".join(map(str, psi1))}

# Well-tempered метадинамика по двум CV
metad: METAD ...
   ARG=phi,psi
   PACE=500             # холм каждые 500 шагов = 1 пс
   HEIGHT=1.2           # начальная высота холма, кДж/моль
   SIGMA=0.35,0.35      # ширина холма по φ₂ и ψ₁, рад (~20°)
   BIASFACTOR=6         # γ = (T + ΔT)/T
   TEMP=298.15          # температура системы, K
   GRID_MIN=-pi,-pi     # смещение хранится на сетке: быстрее, чем суммировать все холмы
   GRID_MAX=pi,pi
   FILE=HILLS           # центры, ширины и высоты всех холмов
...

# Значения CV и смещения каждую 1 пс
PRINT ARG=phi,psi,metad.bias STRIDE=500 FILE=COLVAR
"""
with open("plumed.dat", "w") as f:
    f.write(plumed_dat)

In [ ]:
metad_mdp = """\
integrator    = md           ; leap-frog
dt            = 0.002        ; 2 фс: связи с водородом зафиксированы
nsteps        = 2500000      ; 2500000 × 2 фс = 5 нс

nstxout-compressed = 500     ; кадр в metad.xtc каждую 1 пс
nstenergy     = 500          ; энергии в .edr каждую 1 пс
nstlog        = 5000         ; строка в .log каждые 10 пс

continuation  = yes          ; продолжение NPT из раздела 1.1: скорости, термостат и баростат из npt.cpt
constraints   = h-bonds      ; длины связей X-H пептида фиксированы
constraint-algorithm = lincs ; алгоритм LINCS для связей пептида

cutoff-scheme = Verlet       ; список соседей Верле с буфером
nstlist       = 10           ; список соседей обновляется каждые 10 шагов
coulombtype   = PME          ; электростатика по Эвальду
rcoulomb      = 1.0          ; радиус отсечки прямой части PME, нм
fourierspacing = 0.12        ; шаг сетки PME, нм
pme-order     = 4            ; кубическая интерполяция зарядов на сетку
vdwtype       = Cut-off      ; ЛД отсекается на rvdw
rvdw          = 1.0          ; радиус отсечки ЛД, нм
DispCorr      = EnerPres     ; хвостовая поправка к энергии и давлению
pbc           = xyz          ; периодические граничные условия по трём осям

tcoupl        = V-rescale    ; термостат Бусси
tc-grps       = System       ; термостатируемая группа - все атомы
tau-t         = 0.1          ; время связи с термостатом, пс
ref-t         = 298.15       ; температура, K; та же, что TEMP в plumed.dat
pcoupl          = C-rescale  ; баростат, как в разделе 1.1
pcoupltype      = isotropic  ; бокс масштабируется одинаково по трём осям
tau-p           = 2.0        ; время связи с баростатом, пс
ref-p           = 1.0        ; целевое давление, бар
compressibility = 4.5e-5     ; изотермическая сжимаемость воды при 298 K, 1/бар

gen-vel       = no           ; скорости продолжаются из npt.cpt
"""
with open("metad.mdp", "w") as f:
    f.write(metad_mdp)

In [ ]:
# Запуск симуляции (~ 4 мин)
!/opt/gromacs/bin/gmx grompp -f metad.mdp -c npt.gro -t npt.cpt -p topol.top -o metad.tpr > grompp_metad.out 2>&1
!/opt/gromacs/bin/gmx mdrun -deffnm metad -plumed plumed.dat -ntomp 4 -pin on > mdrun_metad.out 2>&1

In [ ]:
# Сумма всех холмов → F(φ, ψ); для well-tempered PLUMED сам учитывает множитель γ/(γ-1)
!/opt/plumed/bin/plumed --no-mpi sum_hills --hills HILLS --mintozero --bin 72,72 --outfile fes.dat > sum_hills.log 2>&1

F_MAX = 30                                                   # верх цветовой шкалы, кДж/моль
nb = 72
d = np.loadtxt("fes.dat", comments="#")
phi_g = np.rad2deg(d[:nb, 0])                                # φ меняется быстрее всего
psi_g = np.rad2deg(d[::nb, 1])
F = d[:, 2].reshape(nb, nb)                                  # строки - ψ, столбцы - φ
levels = np.arange(0, F_MAX + 1, 2)

t, phi, psi, bias = np.loadtxt("COLVAR", comments="#").T     # время в пс, φ и ψ в рад, смещение в кДж/моль

def unwrap_gaps(a):
    """Угол в градусах; скачок через ±180° размыкаем, чтобы линия не прочерчивала всю ось"""
    a = np.rad2deg(a).copy()
    a[1:][np.abs(np.diff(a)) > 180] = np.nan
    return a

fig = plt.figure(figsize=(12.5, 3.4))
gs = fig.add_gridspec(2, 2, width_ratios=[2.2, 1.2], hspace=0.0, wspace=0.3)
ax_phi = fig.add_subplot(gs[0, 0])
ax_psi = fig.add_subplot(gs[1, 0], sharex=ax_phi)
ax_f = fig.add_subplot(gs[:, 1])

for ax, a, lab in ((ax_phi, phi, "φ, °"), (ax_psi, psi, "ψ, °")):
    ax.plot(t / 1000, unwrap_gaps(a), color="black", lw=0.5)
    ax.axhline(0, color="grey", lw=0.6, ls="--")
    ax.set_ylim(-180, 180)
    ax.set_ylabel(lab)
ax_phi.set_yticks([-90, 0, 90, 180])                         # у стыка панелей по одной подписи с каждой стороны
ax_psi.set_yticks([-180, -90, 0, 90])
ax_phi.tick_params(labelbottom=False)
ax_psi.set_xlim(0, t[-1] / 1000)
ax_psi.set_xlabel("t, нс")

im = ax_f.contourf(phi_g, psi_g, F, levels=levels, cmap="RdYlBu_r", extend="max")
ax_f.contour(phi_g, psi_g, F, levels=levels[::2], colors="black", linewidths=0.3)
ax_f.set_xlim(-180, 180)
ax_f.set_ylim(-180, 180)
ax_f.set_xticks([-180, -90, 0, 90, 180])
ax_f.set_yticks([-180, -90, 0, 90, 180])
ax_f.set_aspect("equal")
ax_f.set_anchor("E")                                         # квадрат к правому краю колонки, вплотную к шкале
ax_f.set_xlabel("φ, °")
ax_f.set_ylabel("ψ, °")
fig.colorbar(im, ax=ax_f, fraction=0.05, pad=0.1, ticks=levels[::2], label="F, кДж/моль")
plt.show()

# 🎲 Глава 2. Метод Монте-Карло

## 2.1 Монте-Карло с нуля

Если величину можно записать как среднее значение некоторой функции по случайной выборке, то её можно оценить, сгенерировав $N$ случайных точек и усреднив по ним:

$$\langle f \rangle \approx \frac{1}{N}\sum_{i=1}^{N} f(x_i).$$

Ошибка среднего по случайной выборке убывает как $1/\sqrt{N}$: чтобы уменьшить её в 10 раз, нужно в 100 раз больше точек. Это медленно. Но в статистической механике средние параметры считаются по координатам всех атомов сразу: для 100 атомов это 300 переменных, и сетка даже из 10 точек на каждую потребовала бы $10^{300}$ узлов. А ошибка Монте-Карло от размерности не зависит вовсе: $1/\sqrt{N}$ и для одной переменной, и для трёхсот. Поэтому для многомерных интегралов случайная выборка не просто удобна, а является единственным работающим способом.

### 2.1.1 Подбрасывание монетки

Начнём с самой простой случайной величины. Подбросим монетку $N$ раз и посчитаем долю орлов и решек. Ответ заранее известен, для  честной монетки доля орлов и решек должна стремиться к 1/2, но как быстро она к нему приходит и совпадает ли результат с ожидаемым разбросом ($1/(2\sqrt{N})$)? 

In [ ]:
# Монетка: доля орлов в серии бросков сходится к 1/2
rng = np.random.default_rng(42)
N = 10000                                         # бросков в серии
SERIES = 5                                        # независимых серий
colors = ["#3B6EA8", "#DC602E", "#6B8E23", "#E0A030", "#7F5A9E"]

n = np.arange(1, N + 1)
fig, ax = plt.subplots(figsize=(8, 3.6))
for k in range(SERIES):
    heads = rng.integers(0, 2, N)                 # 1 - орёл, 0 - решка
    ax.plot(n, np.cumsum(heads) / n, color=colors[k], lw=0.9)
ax.plot(n, 0.5 + 1 / (2 * np.sqrt(n)), color="black", lw=0.8, ls="--")   # ± одно стандартное отклонение
ax.plot(n, 0.5 - 1 / (2 * np.sqrt(n)), color="black", lw=0.8, ls="--")
ax.axhline(0.5, color="grey", lw=0.6)
ax.set_xscale("log")
ax.set_xlim(1, N)
ax.set_ylim(0, 1)
ax.set_xlabel("число бросков N")
ax.set_ylabel("доля орлов")
plt.show()

Ns = [10, 100, 1000, 10000]

df = pd.DataFrame({
    "N бросков":   Ns,
    "орлов":       [heads[:N].sum() for N in Ns],
    "доля орлов":  [heads[:N].mean() for N in Ns],
    "доля решек":  [1 - heads[:N].mean() for N in Ns],
    "1/(2√N)":     [1 / (2 * np.sqrt(N)) for N in Ns],
})
df.round(3)

###  2.1.2 Определение числа $\pi$

Случайные величины можно использовать для решения вполне прикладных задач, например для определения числа $\pi$. Впишем в квадрат со стороной 2 круг радиуса 1. Площадь квадрата 4, площадь круга $\pi$, значит, круг занимает долю $\pi/4$ квадрата. Будем бросать в квадрат случайные точки с равномерным распределением: каждая попадает в круг с вероятностью $\pi/4$, и доля попавших точек после $N$ бросков даёт оценку $\pi \approx 4\,N_{\text{в круге}}/N$.

Для монетки вероятность $p = 1/2$, здесь вероятность $\pi/4 \approx 0.785$, и ошибка оценки убывает по тому же закону: $4\sqrt{p(1-p)/N} \approx 1.64/\sqrt{N}$. Первые два знака $\pi$ получаются уже за тысячу точек, а третий потребует миллиона.

In [ ]:
# Число π: доля случайных точек квадрата [-1, 1]², попавших в единичный круг, равна π/4
rng = np.random.default_rng(7)
N_SHOW = 3000                                     # точек на рисунке
N_MAX = 1_000_000                                 # точек для сходимости

xy = rng.uniform(-1, 1, (N_MAX, 2))
inside = (xy ** 2).sum(axis=1) <= 1.0             # попала ли точка в круг
n = np.arange(1, N_MAX + 1)
pi_est = 4 * np.cumsum(inside) / n                # оценка π после n точек
sigma = 4 * np.sqrt(np.pi / 4 * (1 - np.pi / 4) / n)   # стандартное отклонение оценки

fig, (ax_p, ax_e) = plt.subplots(1, 2, figsize=(12.5, 4.6), gridspec_kw={"width_ratios": [1, 1.4]})
pts = xy[:N_SHOW]
ins = inside[:N_SHOW]
ax_p.scatter(*pts[ins].T, s=3, color="#3B6EA8")
ax_p.scatter(*pts[~ins].T, s=3, color="#DC602E")
a = np.linspace(0, 2 * np.pi, 400)
ax_p.plot(np.cos(a), np.sin(a), color="black", lw=1.0)
ax_p.set_xlim(-1, 1)
ax_p.set_ylim(-1, 1)
ax_p.set_aspect("equal")
ax_p.set_xticks([-1, 0, 1])
ax_p.set_yticks([-1, 0, 1])
ax_p.set_title(f"N = {N_SHOW}: π ≈ {pi_est[N_SHOW - 1]:.4f}")

ax_e.fill_between(n, np.pi - sigma, np.pi + sigma, color="grey", alpha=0.25, lw=0)   # π ± σ
ax_e.plot(n, pi_est, color="#3B6EA8", lw=0.9)
ax_e.axhline(np.pi, color="black", lw=0.8, ls="--")
ax_e.set_xscale("log")
ax_e.set_xlim(10, N_MAX)
ax_e.set_ylim(2.6, 3.7)
ax_e.set_xlabel("число точек N")
ax_e.set_ylabel("оценка π")
plt.tight_layout()
plt.show()

Ns = [10, 100, 1000, 10000, 100000, 1000000]
xy = rng.uniform(-1, 1, (Ns[-1], 2))
inside = (xy ** 2).sum(axis=1) <= 1              # точка внутри единичного круга

df = pd.DataFrame({
    "N точек":      Ns,
    "в круге":      [inside[:N].sum() for N in Ns],
    "оценка π":     [4 * inside[:N].mean() for N in Ns],
    "ошибка":       [abs(4 * inside[:N].mean() - np.pi) for N in Ns],
    "1.64/√N":      [1.64 / np.sqrt(N) for N in Ns],
})
df.round(4)

$N = 10^6$ случайных точек дают π с точностью до третьего знака: ошибка остаётся в пределах $1.64/\sqrt{N}$. Для числа π это крайне неэффективный метод, но тот же приём работает в пространстве любой размерности. Слабое место простого Монте-Карло другое: точки бросаются равномерно, а в статистической механике почти вся вероятность сосредоточена в крошечной области конфигураций с низкой энергией. Равномерная выборка туда почти не попадает, и следующий шаг - выборка по важности.

### 2.1.3 Выборка по важности и распределение Больцмана

Вернёмся к частице на торсионном потенциале из главы 1 и посчитаем её среднюю энергию при 298 K. При этой температуре частица почти всё время сидит на дне потенциальных ям, где $V$ мала, и очень редко оказывается на барьерах. Среднее нужно брать не по всем значениям φ одинаково, а с весом: конфигурации с низкой энергией встречаются часто, с высокой - редко. Этот вес и есть распределение Больцмана $P(\varphi) \propto e^{-V(\varphi)/k_\mathrm{B}T}$, и среднее любой величины $A$ записывается как

$$\langle A \rangle = \frac{\int A(\varphi)\, e^{-\beta V(\varphi)}\, d\varphi}{\int e^{-\beta V(\varphi)}\, d\varphi}, \qquad \beta = \frac{1}{k_\mathrm{B}T}.$$

Как посчитать этот интеграл методом Монте-Карло? Первый способ - тот же, что для числа π: бросаем точки по φ равномерно, а больцмановский вес $w_i = e^{-\beta V(\varphi_i)}$ учитываем потом, как множитель при усреднении. Способ правильный, но расточительный. Точки, попавшие на барьер, получают вес в тысячи раз меньше, чем точки на дне ямы, и в среднем почти не участвуют: мы потратили на них время, а пользы от них нет. Удобная мера этой потери - эффективное число точек $N_{эфф}$: сколько «полноценных» точек стоит наша выборка. Если все веса одинаковы, $N_{эфф} = N$; если весь вес собрала одна точка, $N_{эфф} = 1$. Формула:

$$N_{эфф} = \frac{\left(\sum_i w_i\right)^2}{\sum_i w_i^2}.$$

Второй способ - бросать точки не равномерно, а сразу туда, где частица бывает: генерировать φ с вероятностью $P(\varphi) \propto e^{-\beta V(\varphi)}$. Тогда веса не нужны, и среднее считается как обычное среднее арифметическое по выборке, а каждая точка работает в полную силу. Это называется **выборкой по важности** (importance sampling). Ниже оба способа на одной картинке: слева равномерные точки, размер кружка пропорционален весу; справа точки, сгенерированные по Больцману.

In [ ]:
# Равномерная выборка и выборка по Больцману для частицы на торсионном потенциале
R = 8.314462618e-3                  # кДж/(моль·K)
T = 298.15                          # K
beta = 1 / (R * T)
A, B = 8.0, 4.0                     # V(φ) = A(1 + cos 3φ) + B(1 - cos φ), кДж/моль, как в главе 1

def V(phi):
    return A * (1 + np.cos(3 * phi)) + B * (1 - np.cos(phi))

rng = np.random.default_rng(4)
N = 200                                               # точек в каждой выборке
grid = np.linspace(-np.pi, np.pi, 2001)
deg = np.rad2deg(grid)
p = np.exp(-beta * V(grid))
p /= p.sum() * (grid[1] - grid[0])                    # распределение Больцмана, нормированное по φ в радианах

# Равномерная выборка: каждая точка получает вес exp(-βV)
phi_u = rng.uniform(-np.pi, np.pi, N)
w = np.exp(-beta * V(phi_u))
w /= w.sum()
# Выборка по важности: точки сразу из распределения Больцмана (обратная функция распределения на сетке)
cdf = np.cumsum(p)
cdf /= cdf[-1]
phi_b = np.interp(rng.random(N), cdf, grid)

V_exact = (V(grid) * p).sum() * (grid[1] - grid[0])   # точное <V> численным интегрированием
V_unif = (w * V(phi_u)).sum()                          # взвешенное среднее по равномерной выборке
V_boltz = V(phi_b).mean()                              # простое среднее по больцмановской выборке
ess = 1 / (w ** 2).sum()                               # эффективное число точек равномерной выборки

fig, axes = plt.subplots(1, 2, figsize=(12.5, 3.8), sharey=True)
for ax, x, s, color, title in [
        (axes[0], phi_u, 4 + 3000 * w, "#DC602E", f"равномерно: эффективно {ess:.0f} из {N} точек"),   # размер ∝ вес
        (axes[1], phi_b, np.full(N, 12), "#3B6EA8", f"по Больцману: все {N} точек")]:
    ax.plot(deg, V(grid), color="black", lw=1.2)
    ax.scatter(np.rad2deg(x), V(x), s=s, color=color, edgecolor="black", lw=0.3, alpha=0.8, zorder=3)
    ax.set_xlim(-180, 180)
    ax.set_xticks([-180, -90, 0, 90, 180])
    ax.set_xlabel("φ, °")
    ax.set_title(title)
axes[0].set_ylabel("V, кДж/моль")
plt.show()

print(f"<V> точно:                {V_exact:.2f} кДж/моль")
print(f"<V> равномерно, с весами: {V_unif:.2f} кДж/моль")
print(f"<V> по Больцману:         {V_boltz:.2f} кДж/моль")

Для одного торсиона равномерная выборка ещё работает: из 200 точек полезна примерно четверть, остальные лежат на склонах и барьерах с весом, близким к нулю. Но у системы с множеством торсионных углов вес конфигурации - это произведение весов всех торсионов. Посмотрим, что происходит с долей полезных точек $N_{эфф}/N$ для цепочки из $d$ независимых торсионов.

In [ ]:
# Цепочка из d независимых торсионов: доля полезных точек для равномерной выборки и выборки по Больцману
D = np.arange(1, 31)                                   # число торсионов
N = 100_000                                            # точек в выборке для каждого d
rng = np.random.default_rng(5)

cdf = np.cumsum(p)                                     # функция распределения Больцмана на сетке, как в прошлой ячейке
cdf /= cdf[-1]

ess_unif, ess_boltz = [], []
for d in D:
    # равномерно: вес конфигурации - произведение весов торсионов
    phi = rng.uniform(-np.pi, np.pi, (N, d))
    logw = -beta * V(phi).sum(axis=1)
    w = np.exp(logw - logw.max())                      # вычитаем максимум, чтобы не уйти в 0
    ess_unif.append(w.sum() ** 2 / (w ** 2).sum() / N)
    # по Больцману: каждый торсион из своего распределения; вес = (целевое / распределение выборки) = const
    phi = np.interp(rng.random((N, d)), cdf, grid)
    w = np.ones(N)
    ess_boltz.append(w.sum() ** 2 / (w ** 2).sum() / N)

# Точная доля для одного торсиона: (∫e^{-βV})² / (2π ∫e^{-2βV}); для d торсионов - в степени d
dx = grid[1] - grid[0]
q1 = (np.exp(-beta * V(grid)).sum() * dx) ** 2 / (2 * np.pi * np.exp(-2 * beta * V(grid)).sum() * dx)

fig, ax = plt.subplots(figsize=(8, 3.6))
ax.semilogy(D, ess_boltz, "o", color="#3B6EA8", ms=4, label="по Больцману")
ax.semilogy(D, ess_unif, "o", color="#DC602E", ms=4, label="равномерно, с весами")
ax.semilogy(D, q1 ** D, color="black", lw=0.8, ls="--", label=f"точно: {q1:.2f}$^d$")
ax.axhline(1 / N, color="grey", lw=0.6)
ax.text(D[-1], 1.6 / N, "одна точка из N", ha="right", color="grey")
ax.set_xlabel("число торсионов d")
ax.set_ylabel("доля полезных точек")
ax.set_xlim(0, D[-1] + 1)
ax.set_ylim(0.3 / N, 3)
ax.legend(frameon=False, loc="center right")
plt.show()


Доля полезных точек падает как $0.24^d$: каждый новый торсион отбрасывает три четверти выборки. Уже при $d \approx 8$ из ста тысяч точек весь вклад даёт одна точка из N, и оценка выходит на серую линию - дальше она ничего не значит. Синие точки лежат на единице при любом d, но только потому, что торсионы независимы и для каждого мы заранее построили функцию распределения

Но у выборки по Больцману есть своя трудность: в одном измерении мы сгенерировали точки через функцию распределения на сетке, а для молекулы такой сетки не построить - для этого пришлось бы знать статистическую сумму $Z = \int e^{-\beta V}dx$, то есть уже решить задачу. Нужен способ генерировать конфигурации с вероятностью $\propto e^{-\beta V}$, зная только энергию текущей конфигурации.

## 2.2 Критерий Метрополиса

В 1953 году Метрополис, Розенблюты и Теллеры предложили способ генерировать конфигурации с вероятностью $P(x) \propto e^{-\beta V(x)}$, не зная статистической суммы ([Metropolis et al., 1953](https://doi.org/10.1063/1.1699114)). Вместо независимых точек строится цепь Маркова: каждая следующая конфигурация получается из текущей по простому правилу.

1. Из текущей конфигурации $x$ делаем случайный пробный шаг в $x'$, например поворачиваем торсион на случайный угол из интервала $[-\Delta, \Delta]$.
2. Считаем изменение энергии $\Delta V = V(x') - V(x)$.
3. Принимаем шаг с вероятностью
$$p_{acc} = \min\left(1,\ e^{-\beta \Delta V}\right).$$
Шаг вниз по энергии принимается всегда, шаг вверх - тем реже, чем выше подъём.
4. Если шаг отвергнут, цепочка остаётся в $x$, и эта конфигурация **засчитывается ещё раз**.

Почему цепочка приходит именно к распределению Больцмана? Отношение вероятностей перехода $x \to x'$ и $x' \to x$ по этому правилу равно $e^{-\beta (V(x') - V(x))}$, то есть отношению больцмановских весов. Это условие детального баланса: в равновесии поток $x \to x'$ точно компенсирует поток $x' \to x$. В правило входит только **разность** энергий, поэтому статистическая сумма $Z$ сокращается и знать её не нужно.

Метод Монте-Карло не использует ни силы, ни скорости, ни время. МК-шаг - это не пикосекунда, а одна попытка, и о динамике (скорости переходов, временах жизни) цепочка ничего не говорит. Зато пробные шаги могут быть любыми, лишь бы они были симметричны: шаг $x \to x'$ предлагается так же часто, как $x' \to x$. Размер шага - единственный параметр, который нужно подобрать.


In [ ]:
# Метрополис на торсионном потенциале: три размера пробного шага
R = 8.314462618e-3                  # кДж/(моль·K)
T = 298.15                          # K
beta = 1 / (R * T)
A, B = 8.0, 4.0                     # V(φ) = A(1 + cos 3φ) + B(1 - cos φ), кДж/моль, как в главе 1

def V(phi):
    return A * (1 + np.cos(3 * phi)) + B * (1 - np.cos(phi))

RUNS = [("шаг ±5°",   5.0,  "#3B6EA8"),
        ("шаг ±40°",  40.0,  "#6B8E23"),
        ("шаг ±180°", 180.0, "#DC602E")]                 # (заголовок, максимальный шаг в градусах, цвет)
N_BALLS = 30                        # независимых цепочек Маркова в каждой панели
STEPS_PER_FRAME = 20                # МК-шагов между кадрами анимации
N_FRAMES = 150                      # 3000 МК-шагов всего

rng = np.random.default_rng(11)
M = len(RUNS)
dmax = np.deg2rad([d for _, d, _ in RUNS])[:, None]

# Все цепочки стартуют в левой яме
x = np.full((M, N_BALLS), np.deg2rad(-60.0))
traj = np.empty((N_FRAMES + 1, M, N_BALLS))
acc_rate = np.zeros((N_FRAMES + 1, M))                  # доля принятых шагов к каждому кадру
traj[0] = x
n_acc = np.zeros(M)
for f in range(1, N_FRAMES + 1):
    for _ in range(STEPS_PER_FRAME):
        x_new = x + rng.uniform(-1, 1, x.shape) * dmax  # 1. пробный шаг
        x_new = (x_new + np.pi) % (2 * np.pi) - np.pi   #    периодичность угла
        dV = V(x_new) - V(x)                            # 2. изменение энергии
        acc = rng.random(x.shape) < np.exp(-beta * dV)  # 3. принять с вероятностью min(1, e^{-βΔV})
        x = np.where(acc, x_new, x)                     # 4. если отвергнут - остаёмся на месте и считаем точку ещё раз
        n_acc += acc.sum(axis=1)
    traj[f] = x
    acc_rate[f] = n_acc / (f * STEPS_PER_FRAME * N_BALLS)

grid = np.linspace(-np.pi, np.pi, 361)
deg = np.rad2deg(grid)
edges = np.linspace(-180, 180, 37)
p_ref = np.exp(-beta * V(grid))
p_ref /= p_ref.sum() * (deg[1] - deg[0])                # Больцман при 298 K, нормирован по φ в градусах

fig, axes = plt.subplots(2, M, figsize=(12.5, 5.2), sharex=True, gridspec_kw={"height_ratios": [1.3, 1]})
balls, hists, titles = [], [], []
for k, (title, _, color) in enumerate(RUNS):
    ax, axh = axes[0, k], axes[1, k]
    ax.plot(deg, V(grid), color="black", lw=1.2)
    balls.append(ax.scatter([], [], s=40, color=color, edgecolor="black", lw=0.4, zorder=3))
    titles.append(ax.set_title(title))
    ax.set_ylim(-1, 25)
    axh.plot(deg, p_ref, color="black", lw=0.8, ls="--")
    hists.append(axh.stairs(np.zeros(len(edges) - 1), edges, fill=True, color=color, alpha=0.6))
    axh.set_ylim(0, 1.6 * p_ref.max())
    axh.set_xlim(-180, 180)
    axh.set_xticks([-180, -90, 0, 90, 180])
    axh.set_xlabel("φ, °")
axes[0, 0].set_ylabel("V, кДж/моль")
axes[1, 0].set_ylabel("P(φ)")
step_text = axes[0, 0].text(0.03, 0.92, "", transform=axes[0, 0].transAxes)

def update(f):
    for k, (title, _, _) in enumerate(RUNS):
        balls[k].set_offsets(np.column_stack([np.rad2deg(traj[f, k]), V(traj[f, k]) + 0.8]))   # шарик чуть над кривой
        H, _ = np.histogram(np.rad2deg(traj[:f + 1, k].ravel()), bins=edges, density=True)
        hists[k].set_data(H)
        titles[k].set_text(f"{title} (принято: {100 * acc_rate[f, k]:.0f} %)" if f > 0 else title)
    step_text.set_text(f"МК-шаг {f * STEPS_PER_FRAME}")
    return balls + hists + titles + [step_text]

anim = animation.FuncAnimation(fig, update, frames=N_FRAMES + 1, interval=60, blit=False)
plt.close(fig)
with plt.rc_context({"savefig.dpi": 80, "animation.embed_limit": 40}):   # все 151 кадр, ~20 МБ в выводе
    html = anim.to_jshtml()
HTML(html)

Все три цепочки подчиняются одному и тому же правилу и в пределе дают одно и то же распределение Больцмана, но приходят к нему с разной скоростью.

- **Маленький шаг (±5°)** принимается почти всегда, но цепочка ползёт по яме как при диффузии и за 3000 шагов ни разу не перебирается в правую яму. Высокая доля принятых шагов сама по себе ничего не значит.
- **Большой шаг (±180°)** перебрасывает шарик в любую точку круга, в том числе сразу через барьер. Большинство таких прыжков попадает на склоны и отвергается, но удачные прыжки сразу переносят цепочку в другую яму.
- **Промежуточный шаг (±40°)** даёт долю принятых шагов около 40 %, и распределение быстро приходит к правильному. Обычно размер шага подбирают так, чтобы принималось 20-50 % попыток.

В отличие от молекулярной динамики, в методе Монте-Карло частица может перепрыгивать через барьер, не поднимаясь на него: пробный шаг не обязан быть физическим движением. 

## 2.3 Сборка белков методом Монте-Карло

Монте-Карло не обязан двигать атомы по декартовым координатам. Для белков естественные переменные - торсионные углы остова φ и ψ: длины связей и валентные углы почти постоянны, и вся конформация цепи определяется набором торсионов. Именно так работает Rosetta: пробный шаг меняет торсионы одного или нескольких остатков, энергия пересчитывается, и шаг принимается или отвергается по критерию Метрополиса.

Соберём α-спираль из поли-аланина длиной 12 остатков, стартуя с вытянутой цепи. Пробный шаг самый простой: сдвигаем φ и ψ случайного остатка на величину до ±D и сравним три размера шага, как в разделе 2.2. Такой шаг симметричен, поэтому критерий Метрополиса применяется в чистом виде. Предпочтения карты Рамачандрана при этом не теряются: они уже заложены в функцию энергии Rosetta ref2015, и шаг в запрещённую область будет отвергнут из-за высокой энергии. Энергия считается в условных единицах REU (Rosetta Energy Units), температура тоже задаётся в REU.

In [ ]:
main_path = os.path.join(REPO, "Занятия", "Занятие_9")
work_path = os.path.join(main_path, "helix_mc")
os.makedirs(work_path, exist_ok=True)
os.chdir(work_path)

In [ ]:
# Поли-Ala из 12 остатков: вытянутая цепь на старте, идеальная α-спираль как эталон
from pyrosetta import pose_from_sequence, get_fa_scorefxn
from pyrosetta.rosetta.core.scoring import CA_rmsd

SEQ = "A" * 12
N_RES = len(SEQ)
sf = get_fa_scorefxn()                                # полноатомная функция энергии ref2015, энергия в REU

def chain(phi, psi):
    """Поли-Ala с одинаковыми φ, ψ у всех остатков"""
    pose = pose_from_sequence(SEQ)
    for i in range(1, N_RES + 1):
        pose.set_phi(i, phi)
        pose.set_psi(i, psi)
        pose.set_omega(i, 180.0)
    return pose

ideal = chain(-57.0, -47.0)                            # идеальная α-спираль - эталон для RMSD
start = chain(-150.0, 150.0)                           # вытянутая β-цепь - старт МК
print(f"E(вытянутая цепь) = {sf(start):6.1f} REU")
print(f"E(α-спираль)      = {sf(ideal):6.1f} REU")

In [ ]:
# Метрополис по торсионам остова при kT = 0.5 REU: три цепочки, различается только размер пробного шага
kT = 0.5                                               # температура в единицах энергии Rosetta (REU)
STEPS_DEG = [30, 60, 180]                              # полуширина шага по φ и ψ, °; ±180° = случайные новые углы
N_STEPS = 160_000                                      # МК-шагов в цепочке
EVERY = 400                                            # запись энергии и RMSD каждые 400 шагов
FRAME_EVERY = 8000                                     # структура в траекторию каждые 8000 шагов: 21 кадр
rng = np.random.default_rng(1)

def frame_lines(pose):
    """Координаты позы как строки ATOM для многомодельного PDB"""
    pose.dump_pdb("_frame.pdb")
    return [l for l in open("_frame.pdb") if l.startswith("ATOM")]

results = {}
for D in STEPS_DEG:
    pose = start.clone()
    E = sf(pose)
    log, models, n_acc = [], [], 0
    for step in range(N_STEPS + 1):
        if step % EVERY == 0:
            log.append((step, E, CA_rmsd(pose, ideal, 2, N_RES - 1)))
        if step % FRAME_EVERY == 0:
            models.append(frame_lines(pose))
        i = int(rng.integers(1, N_RES + 1))                         # случайный остаток
        old = (pose.phi(i), pose.psi(i))
        new = (np.array(old) + rng.uniform(-D, D, 2) + 180) % 360 - 180   # 1. пробный шаг: сдвиг φ и ψ до ±D
        pose.set_phi(i, new[0])
        pose.set_psi(i, new[1])
        E_new = sf(pose)                                            # 2. новая энергия
        if rng.random() < np.exp(-(E_new - E) / kT):                # 3. критерий Метрополиса
            E = E_new
            n_acc += 1
        else:                                                       # 4. отказ - возвращаем старые углы
            pose.set_phi(i, old[0])
            pose.set_psi(i, old[1])
    results[D] = np.array(log)
    with open(f"polyA_step{D}.pdb", "w") as f:                      # траектория МК: кадр = модель
        for k, lines in enumerate(models, 1):
            f.write(f"MODEL     {k}\n" + "".join(lines) + "ENDMDL\n")
        f.write("END\n")
    print(f"шаг ±{D:3d}°: E = {E:6.1f} REU, RMSD Cα = {log[-1][2]:.1f} Å, принято {100 * n_acc / N_STEPS:.0f} % шагов, "
          f"{len(models)} кадров -> polyA_step{D}.pdb")

In [ ]:
# Энергия и RMSD по ходу МК и их распределения по всем записанным шагам
colors = dict(zip(STEPS_DEG, ["#3B6EA8", "#6B8E23", "#DC602E"]))

fig, axes = plt.subplots(2, 2, figsize=(12.5, 6.4), gridspec_kw={"width_ratios": [2.2, 1]})
for D, log in results.items():
    step, E, rmsd = log.T
    lab = f"шаг ±{D}°"
    axes[0, 0].plot(step / 1000, E, color=colors[D], lw=0.7, label=lab)
    axes[1, 0].plot(step / 1000, rmsd, color=colors[D], lw=0.7)
    axes[0, 1].hist(E, bins=40, color=colors[D], alpha=0.5, orientation="horizontal", label=lab)
    axes[1, 1].hist(rmsd, bins=40, color=colors[D], alpha=0.5, orientation="horizontal")
for ax in axes[0]:
    ax.axhline(sf(ideal), color="black", lw=0.8, ls="--")           # энергия идеальной спирали
axes[0, 0].set_ylabel("E, REU")
axes[1, 0].set_ylabel("RMSD Cα, Å")
axes[1, 0].set_xlabel("МК-шаг, тыс.")
axes[1, 1].set_xlabel("число кадров")
for ax in axes[:, 0]:
    ax.set_xlim(0, N_STEPS / 1000)
for row in axes:
    row[1].sharey(row[0])
    row[1].tick_params(labelleft=False)
axes[0, 0].legend(frameon=False, loc="upper right")
plt.tight_layout()
plt.show()

In [ ]:
show("polyA_step30.pdb", "polyA_step60.pdb", "polyA_step180.pdb",
     cmd="hide everything; show cartoon; show sticks, name N+CA+C+O; "
         "color marine, polyA_step30 and elem C; color forest, polyA_step60 and elem C; color firebrick, polyA_step180 and elem C; "
         "intra_fit polyA_step30 and name CA; intra_fit polyA_step60 and name CA; intra_fit polyA_step180 and name CA; orient")

# 💊 Глава 3. Дизайн лекарств и молекулярный докинг

## 3.1 Малая молекула как лекарственное вещество

Лекарство в организме проходит длинный путь от приёма таблетки до связывания с мишенью. Молекула, которая прочно садится в карман белка в пробирке, бесполезна, если не доходит до этого кармана в организме.

Обычно к лекарству применяются следующие требования:

- **растворимость** в воде - иначе вещество не выйдет из таблетки;
- **проницаемость** через липидный бислой мембраны кишечника - для этого молекула должна быть достаточно липофильной;
- **метаболическая устойчивость** - не разрушиться за первый проход через печень;
- **сродство к мишени** - высокая константа связывания;
- **селективность** - взаимодействует только с мишенью.

Растворимость и проницаемость тянут в противоположные стороны: полярная молекула хорошо растворяется, но плохо проходит мембрану, липофильная - наоборот. Удачное лекарство находится в узком окне между этими требованиями, и границы этого окна можно оценить по уже существующим препаратам для этой мишени.

### 3.1.1 Правило Липинского

Кристофер Липински определил, что подавляющее большинство лекарственных препаратов, дошедших до второй фазы клинических испытаний, укладываются в четыре простых ограничения - **правило пяти** (все пороги кратны пяти)  [Lipinski et al., 1997](https://doi.org/10.1016/S0169-409X(96)00423-1):

| Дескриптор | Порог |
|---|---|
| Молекулярная масса, MW | ≤ 500 Да |
| Липофильность, logP | ≤ 5 |
| Доноров водородной связи, HBD | ≤ 5 |
| Акцепторов водородной связи, HBA | ≤ 10 |

Нарушение **двух и более** условий - признак того, что вещество плохо всасывается при приёме внутрь. Позже к ним добавили правило Вебера: число вращаемых связей ≤ 10 и топологическая поверхность TPSA ≤ 140 Å².

Это не закон природы, а статистика по историческому набору препаратов. Правило не применяют к антибиотикам, противогрибковым препаратам, пептидам и всему, что попадает в организм не через желудок (Например, циклоспорин нарушает три условия из четырёх и остаётся отличным иммунодепрессантом). Но для виртуального скрининга правило Липинского является дешёвым первичным фильтром, с использованием которого отбрасываются заведомо безнадёжные молекулы.

### 3.1.2 Химические базы данных

Источники молекул потенциальных (или подтверждённых) лекарственных препаратов:

- **[ZINC](https://zinc.docking.org/)** - главная база для виртуального скрининга: химические соединения, уже подготовленные для расчёта (протонированы при pH 7, с 3D-координатами и стереоизомерами).
- **[ChEMBL](https://www.ebi.ac.uk/chembl/)** - около 2.5 млн соединений с **измеренной** биологической активностью ($K_i$, $IC_{50}$), более 20 млн измерений из литературы. Используется как обучающая выборка и эталон для проверки протокола.
- **[DrugBank](https://go.drugbank.com/)** - около 15 тысяч одобренных и экспериментальных лекарств с мишенями, механизмом действия и фармакокинетикой.
- **[PubChem](https://pubchem.ncbi.nlm.nih.gov/)** - около 120 млн соединений, самая полная база по химии.
- **[PDB](https://www.rcsb.org/)** - около 230 тысяч структур белков и их комплексов с лигандами, откуда берут мишень и стартовую позу.

Виртуальный скрининг устроен воронкой: миллионы молекул → фильтры по свойствам (правило Липинского, PAINS) → докинг, несколько секунд на молекулу → сотня лучших по скору → визуальный отбор (→ молекулярно-динамическое моделирование белка с лигандом) → эксперимент. Каждая ступень на порядки дешевле следующей, поэтому дешёвые фильтры ставят первыми.

**PAINS** (Pan-Assay Interference Compounds) - структурные фрагменты, которые дают сигнал почти в любом биологическом тесте не из-за связывания с мишенью, а из-за агрегации, окисления или флуоресценции ([Baell, Holloway, 2010](https://doi.org/10.1021/jm901137j)). 

## 3.2 Молекулярный докинг

Молекулярный докинг предсказывает, как малая молекула (лиганд) связывается с белком: в каком месте (позе), в какой пространственной ориентации и конформации и насколько прочно. Это главный вычислительный инструмент ранних стадий поиска лекарств: виртуальный скрининг пропускает через докинг миллионы молекул, чтобы выбрать для синтеза и эксперимента несколько сотен.

В качестве тестовой задачи для молекулярного докинга можно попробовать запустить **повторный докинг**: взять готовую кристаллическую структуру комплекса, вынуть из неё лиганд, поставить его в случайное положение и посмотреть, вернёт ли программа исходную позу. Правильный ответ известен заранее, поэтому качество видно сразу: успехом считается RMSD лиганда до кристаллической позы меньше 2 Å у решения с лучшей энергией.

Наша модельная система - **трипсин с бензамидином** ([PDB 3PTB](https://www.rcsb.org/structure/3PTB), разрешение 1.7 Å). Трипсин - сериновая протеаза, которая режет белки после остатков лизина и аргинина. Их положительно заряженные боковые цепи входят в глубокий карман S1 и образуют солевой мостик с Asp189 на его дне. Бензамидин копирует боковую цепь аргинина: амидиновая группа взаимодействует с Asp189, бензольное кольцо заполняет карман. Это классический конкурентный ингибитор трипсина ($K_i$ = 18 мкМ). 

### 3.2.1 Подготовка структуры

Кристаллическая структура из PDB не готова к расчёту. В файле 3PTB кроме белка лежат бензамидин, ион кальция и 60 молекул воды, а водородов нет вовсе. Оставим только записи `ATOM` - это белок; всё остальное (`HETATM`) выбросим. Координаты кристаллического бензамидина сохраним отдельно, только для проверки ответа.

Лиганд готовим из SMILES: RDKit строит 3D-геометрию, оптимизирует её силовым полем MMFF и записывает в файл SDF со связями, зарядами и водородами. Из такого файла Rosetta сама строит **тип остатка**: назначает атомам свои типы (`aroC`, `NH2O`, `Hpol`...) и парциальные заряды. Протокол RosettaLigand требует, чтобы лиганд был **отдельной цепью X**.

Важна **протонированная форма**. Бензамидин - сильное основание ($pK_a$ ≈ 11.6), в воде он существует как катион бензамидиния, и именно заряд +1 отвечает за солевой мостик с Asp189. Поэтому в SMILES пишем `NC(=[NH2+])c1ccccc1`, а не нейтральную форму.

In [ ]:
main_path = os.path.join(REPO, "Занятия", "Занятие_9")
work_path = os.path.join(main_path, "rosetta_docking")
os.makedirs(work_path, exist_ok=True)
os.chdir(work_path)

In [ ]:
# Трипсин из PDB 3PTB: оставляем только белок (записи ATOM); кристаллический бензамидин - отдельно, для проверки
import urllib.request

if not os.path.exists("3ptb.pdb"):
    urllib.request.urlretrieve("https://files.rcsb.org/download/3PTB.pdb", "3ptb.pdb")

protein, ben_xtal = [], []
for line in open("3ptb.pdb"):
    record, resname = line[:6], line[17:20].strip()
    if record == "ATOM  ":
        protein.append(line)                               # белок
    elif record == "HETATM" and resname == "BEN":
        ben_xtal.append(line)                              # кристаллическая поза лиганда - эталон

with open("trypsin.pdb", "w") as out:
    out.writelines(protein + ["TER\nEND\n"])
with open("ben_xtal.pdb", "w") as out:
    out.writelines(ben_xtal + ["END\n"])

print(f"белок: {len(protein)} атомов -> trypsin.pdb; кристаллический бензамидин: {len(ben_xtal)} атомов -> ben_xtal.pdb")

In [ ]:
show("3ptb.pdb", "trypsin.pdb", wrap=False)

In [ ]:
# Лиганд из SMILES: 3D-геометрия (ETKDG), оптимизация MMFF, запись в SDF с водородами и зарядами
SMILES = "NC(=[NH2+])c1ccccc1"           # катион бензамидиния
LIG_NAME = "BEN"                         # трёхбуквенный код остатка в PDB

mol = Chem.AddHs(Chem.MolFromSmiles(SMILES))
AllChem.EmbedMolecule(mol, randomSeed=1)
AllChem.MMFFOptimizeMolecule(mol)
mol.SetProp("_Name", LIG_NAME)           # имя молекулы станет именем типа остатка в Rosetta

with Chem.SDWriter("ligand.sdf") as w:
    w.write(mol)

heavy = sum(a.GetAtomicNum() > 1 for a in mol.GetAtoms())
print(f"{LIG_NAME}: {mol.GetNumAtoms()} атомов ({heavy} тяжёлых), формальный заряд {Chem.GetFormalCharge(mol):+d}")
Draw.MolToImage(Chem.RemoveHs(mol), size=(250, 180))

In [ ]:
# Тип остатка из SDF добавляем в набор типов позы, лиганд - новая цепь X
pyrosetta.init("-mute all")
from pyrosetta.rosetta.core.chemical.sdf import MolFileIOReader
from pyrosetta.rosetta.core.conformation import ResidueFactory

def add_ligand(pose, sdf_file):
    """Строим тип остатка из SDF (типы атомов, заряды, связи) и пришиваем лиганд к позе отдельной цепью X"""
    rts = pose.residue_type_set_for_pose()
    molecule = MolFileIOReader().parse_file(sdf_file)[1]                   # первая молекула файла
    new_type = molecule.convert_to_ResidueType(rts.atom_type_set(), rts.element_set(), rts.mm_atom_type_set())
    pose_rts = pose.conformation().modifiable_residue_type_set_for_conf()
    pose_rts.add_base_residue_type(new_type)
    pose.conformation().reset_residue_type_set_for_conf(pose_rts)
    residue = ResidueFactory.create_residue(pose.residue_type_set_for_pose().name_map(new_type.name()))
    pose.append_residue_by_jump(residue, 1, "", "", True)                 # True - начать новую цепь
    n = pose.total_residue()
    pose.pdb_info().set_resinfo(n, "X", 1)                                 # цепь X, номер 1
    pose.pdb_info().obsolete(False)
    return n

native = pyrosetta.pose_from_file("trypsin.pdb")        # белок с достроенными водородами
LIG = add_ligand(native, "ligand.sdf")                  # лиганд - последний остаток
info = native.pdb_info()
lig = native.residue(LIG)
print(f"остатков в позе: {native.total_residue()}, лиганд: остаток {LIG}, цепь {info.chain(LIG)}, "
      f"{lig.name3()}, {lig.natoms()} атомов, суммарный заряд {sum(lig.atomic_charge(i) for i in range(1, lig.natoms() + 1)):+.2f}")

# Как Rosetta типизировала атомы лиганда (атомы идут в том же порядке, что в SDF)
pd.DataFrame({"атом":  [lig.atom_name(i).strip() for i in range(1, lig.natoms() + 1)],
              "тип":   [lig.atom_type(i).name() for i in range(1, lig.natoms() + 1)],
              "заряд": [round(lig.atomic_charge(i), 3) for i in range(1, lig.natoms() + 1)]}).T

Оба азота амидиниевой группы получили тип `NH2O` (sp²-азот с двумя полярными водородами), суммарный заряд ровно +1. 

Куда ставить лиганд? Кристаллического положения у нас больше нет, поэтому **центр кармана** задаём по атомам белка: середина между карбоксилатом Asp189 на дне кармана S1 и гидроксилом каталитического Ser195 у его входа. Этого достаточно: алгоритм поиска сам обследует куб 7 Å вокруг стартовой точки.

In [ ]:
# Центр кармана S1 из атомов белка; лиганд переносим в него и смотрим на старт
from pyrosetta.rosetta.core.id import AtomID
from pyrosetta.rosetta.numeric import xyzVector_double_t

def atom_xyz(pose, pdb_num, atom, chain="A"):
    return np.array(pose.residue(pose.pdb_info().pdb2pose(chain, pdb_num)).xyz(atom))

ASP189_O = np.array([atom_xyz(native, 189, "OD1"), atom_xyz(native, 189, "OD2")])
CENTER = (ASP189_O.mean(axis=0) + atom_xyz(native, 195, "OG")) / 2

NAT = native.residue(LIG).natoms()
HEAVY = [i for i in range(1, NAT + 1) if native.residue(LIG).atom_type(i).element() != "H"]

def lig_xyz(pose):
    return np.array([list(pose.residue(LIG).xyz(i)) for i in range(1, NAT + 1)])

def set_lig_xyz(pose, xyz):
    for i, p in enumerate(xyz, start=1):
        pose.set_xyz(AtomID(i, LIG), xyzVector_double_t(*p))

xyz = lig_xyz(native)
set_lig_xyz(native, xyz - xyz[[i - 1 for i in HEAVY]].mean(axis=0) + CENTER) 
native.dump_pdb("start.pdb")

# Остатки кармана
lig_c = native.residue(LIG).nbr_atom_xyz()
pocket = [i for i in range(1, LIG) if native.residue(i).nbr_atom_xyz().distance(lig_c) < 12.0]
print("центр кармана:", CENTER.round(2))
print("карман:", ", ".join(f"{native.residue(i).name3()}{info.number(i)}" for i in pocket))

resi = "+".join(str(info.number(i)) for i in pocket)
show("start.pdb", "ben_xtal.pdb",
     cmd="hide everything; show cartoon, start and polymer; color grey70, start and polymer; "
         "show sticks, start and chain X; color orange, start and chain X and elem C; "
         "show sticks, ben_xtal; color green, ben_xtal and elem C; "
         f"show sticks, start and (resi {resi}) and sidechain; color marine, start and (resi {resi}) and elem C; "
         "zoom chain X, 5")

### 3.2.2 Функция оценки и алгоритм поиска

Любая программа докинга состоит из двух частей.

- **Функция оценки** ставит каждой позе лиганда число, которое должно коррелировать с энергией связывания. В Rosetta это сумма физических и статистических термов: притяжение и отталкивание Леннард-Джонса (`fa_atr`, `fa_rep`), десольватация (`fa_sol`), электростатика (`fa_elec`), водородные связи (`hbond_*`) и энергия ротамеров боковых цепей (`fa_dun`). Энергия связывания считается как разность $\Delta E = E(\text{комплекс}) - E(\text{белок}) - E(\text{лиганд})$ в единицах REU.
- **Алгоритм поиска**. Пространство огромно: три координаты центра лиганда, три угла ориентации, подвижные торсионы лиганда и боковые цепи аминокислот в кармане. Полный перебор невозможен, и почти все программы докинга используют Монте-Карло с критерием Метрополиса, генетические алгоритмы или их комбинации.

### 3.2.3 Протокол RosettaLigand

RosettaLigand ([Meiler, Baker, 2006](https://doi.org/10.1002/prot.21086); [DeLuca, Khar, Meiler, 2015](https://doi.org/10.1371/journal.pone.0132508)) ищет позу лиганда в несколько стадий, от грубой к точной:

1. **Transform** - Монте-Карло по положению и ориентации жёсткого лиганда в кубе 7 Å вокруг стартовой точки. Энергия считается по заранее рассчитанной сетке, поэтому шаг стоит доли миллисекунды. Каждый шаг - сдвиг до 0.2 Å и поворот до 20°, принятие по критерию Метрополиса.
2. **HighResDocker** - 6 циклов Монте-Карло с малыми сдвигами лиганда, переупаковкой боковых цепей кармана (ротамеры, тоже через Метрополис) и минимизацией. Отталкивание смягчено, чтобы лиганд мог протиснуться мимо боковых цепей.
3. **FinalMinimizer** - градиентная минимизация лиганда, боковых цепей и остова вокруг него с полной функцией энергии.
4. **InterfaceScoreCalculator** - энергия связывания: комплекс сравнивается с белком и лигандом, разведёнными далеко друг от друга.

Протокол описывается на языке RosettaScripts (XML). Каждый запуск даёт одно решение; запустим 100 независимых решений из случайных стартовых поз.

In [ ]:
# Протокол RosettaLigand описан на языке RosettaScripts в tools/ligand_dock.xml
from pyrosetta.rosetta.protocols.rosetta_scripts import XmlObjects

PROTOCOL_XML = os.path.join(REPO, "tools", "ligand_dock.xml")
print(open(PROTOCOL_XML).read())

xml = XmlObjects.create_from_file(PROTOCOL_XML)
STAGES = [xml.get_mover(name) for name in ["transform", "high_res_docker", "final", "add_scores"]]

In [ ]:
# Докинг из случайных стартовых поз: лиганд поворачиваем произвольно и сдвигаем до 2 Å от центра кармана
from scipy.spatial.transform import Rotation
from pyrosetta.rosetta.core.pose import Pose, getPoseExtraScore

N_DECOYS = 100                      # число независимых решений
SHIFT = 2.0                         # максимальный сдвиг стартовой позы от центра кармана, Å
rng = np.random.default_rng(1)

def random_start(pose):
    xyz = lig_xyz(pose)
    c = xyz[[i - 1 for i in HEAVY]].mean(axis=0)
    rot = Rotation.random(random_state=rng.integers(1 << 31))
    v = rng.normal(size=3)
    v *= rng.uniform(0, SHIFT) / np.linalg.norm(v)
    set_lig_xyz(pose, rot.apply(xyz - c) + CENTER + v)

decoys = []
for k in range(N_DECOYS):
    pose = native.clone()
    random_start(pose)
    for stage in STAGES:
        stage.apply(pose)
    decoys.append((getPoseExtraScore(pose, "interface_delta_X"), pose))
    if (k + 1) % 20 == 0:
        print(f"готово {k + 1} из {N_DECOYS}")

decoys.sort(key=lambda d: d[0])                          # от лучшей энергии связывания к худшей

df = pd.DataFrame({"ранг": np.arange(1, len(decoys) + 1), "ΔE, REU": [d[0] for d in decoys]})
df.to_csv("decoys.csv", index=False)

print(f"лучшее по энергии: ΔE = {df.loc[0, 'ΔE, REU']:.1f} REU, худшее: {df['ΔE, REU'].iloc[-1]:.1f} REU")
df.head(10).round(2)

In [ ]:
# Десять лучших и десять худших решений - каждое целиком, с белком: боковые цепи и петля у каждого свои
def dump_pose(pose, name):
    """Поза целиком (белок + лиганд) в PDB; имя файла станет именем объекта в просмотрщике"""
    pose.dump_pdb(name + ".pdb")
    return name

best10  = [dump_pose(d[1], f"top_{i:02d}") for i, d in enumerate(decoys[:10], 1)]
worst10 = [dump_pose(d[1], f"bot_{i:02d}") for i, d in enumerate(decoys[-10:], 1)]

def pocket_cmd(obj, color):
    """Лиганд палочками, остатки в 4.5 Å от него - линиями; углероды в цвет группы"""
    return (f"show sticks, {obj} and chain X; "
            f"show lines, {obj} and (byres ({obj} within 4.5 of ({obj} and chain X))) and not chain X; "
            f"color {color}, {obj} and elem C")

cmd = ["hide everything", f"show cartoon, {best10[0]} and polymer", f"color grey70, {best10[0]} and polymer"]
cmd += [pocket_cmd(n, "marine") for n in best10]
cmd += [pocket_cmd(n, "firebrick") for n in worst10]
cmd.append(f"zoom {best10[0]} and chain X, 6")

print("синие - 10 лучших по энергии, красные - 10 худших; у каждого объекта свой карман")
for i, d in enumerate(decoys[:10], 1):
    print(f"  top_{i:02d}: ΔE = {d[0]:6.1f} REU")
for i, d in enumerate(decoys[-10:], 1):
    print(f"  bot_{i:02d}: ΔE = {d[0]:6.1f} REU")

show("trypsin.pdb", "ben_xtal.pdb", *[n + ".pdb" for n in best10 + worst10], cmd="; ".join(cmd))

Ограничения метода молекулярного докинга:
- **Белок почти жёсткий.** Подвижны только боковые цепи кармана и небольшой кусок остова. Если в реальном комплексе белок структурно перестраивается, докинг такой позы не найдёт и пропустит хорошего кандидата.

- **Скоринг-функция - не свободная энергия.** В ΔE нет энтропии лиганда (при связывании молекула теряет вращательные и конформационные степени свободы) и почти явного растворителя. Поэтому ранжирование разных молекул по ΔE работает заметно хуже, чем поиск позы для одной: корреляция скоров докинга с экспериментальными $K_i$ редко бывает выше 0.5.

Поэтому докинг - фильтр, а не предсказатель: он сокращает миллионы молекул до сотен, а дальше идут более дорогие методы и расчёты свободной энерги. 

# 📚 Дополнительные материалы

**Улучшенный сэмплинг и свободная энергия**:
* [PLUMED](https://www.plumed.org) - метадинамика, umbrella sampling, коллективные переменные; [PLUMED Masterclass](https://www.plumed.org/masterclass) - курс с разбором метадинамики на дипептиде аланина
* [PLUMED-NEST](https://www.plumed-nest.org) - входные файлы опубликованных расчётов, можно взять готовый `plumed.dat` под свою задачу
* Обзоры: [Valsson, Tiwary, Parrinello, 2016](https://doi.org/10.1146/annurev-physchem-040215-112229) - методы с коллективными переменными; [Bussi, Laio, 2020](https://doi.org/10.1038/s42254-020-0153-0) - метадинамика в одном обзоре

**Монте-Карло**:
* [Metropolis et al., 1953](https://doi.org/10.1063/1.1699114) - оригинальная статья с критерием
* Frenkel, Smit, *Understanding Molecular Simulation* - главы 3 и 5: Монте-Карло и методы свободной энергии
* [Rosetta](https://www.rosettacommons.org); [PyRosetta](https://www.pyrosetta.org); [PyRosetta Notebooks](https://rosettacommons.github.io/PyRosetta.notebooks/) - учебные ноутбуки 

**Докинг и дизайн лекарств**:
* [RosettaLigand](https://doi.org/10.1371/journal.pone.0132508) - протокол, который мы использовали
* [AutoDock Vina](https://vina.scripps.edu) и [GNINA](https://github.com/gnina/gnina) - быстрые программы докинга для виртуального скрининга